# Лабораторная работа №6: Построение и оценка VLM-пайплайна

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.


## Цели обучения

После работы студент должен уметь:

- Запустить quantized VLM 2–4B.
- Создать evaluation JSONL.
- Сравнить prompts, resolution и decoding; разметить ошибки.


## Теоретическая часть

VLM связывает visual encoder, projector и language model. Оценка должна разделять captioning, VQA, OCR и structured extraction, фиксировать prompt, decoding и ошибки: hallucination, counting, spatial relations.

![Пример изображения для Visual Question Answering](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/vqa-example.png)


### Математическая постановка

Для открытых ответов одной exact match недостаточно. Используют нормализованный EM/F1, judge rubric и ручную разметку ошибок. Hallucination rate: $H=N_{unsupported}/N_{claims}$.


### Материалы

- [Transformers multimodal chat templates](https://huggingface.co/docs/transformers/chat_templating_multimodal)
- [VLMEvalKit](https://github.com/open-compass/VLMEvalKit)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практическое задание

1. Запустить quantized VLM 2–4B.
2. Создать evaluation JSONL.
3. Сравнить prompts, resolution и decoding; разметить ошибки.

**Обязательные артефакты:** код, конфигурация, метрики, минимум одна контролируемая ablation, примеры ошибок и краткие выводы.


### Профиль вычислений

- **Основной режим:** VLM 4B NF4 inference, одно изображение.
- **Ограничение данных:** 20 evaluation examples: 5 на каждый из четырёх типов задач.
- **Fallback:** VLM 2B NF4 с тем же API.
- Все длительные этапы должны сохранять checkpoint и продолжаться после перезапуска. Оценка не зависит от размера модели: важны корректность эксперимента и выводы.


**Оценивание.** Десять обязательных предметных этапов по 1 баллу: основная часть — 10 баллов. Творческий бонус — отдельно 1 балл, не заменяет обязательные этапы. Полный режим оценивается по реальным данным и инференсу; `FAST_DEV_RUN=True` служит только smoke-проверкой, его показатели не являются отчётными.


## Реализация


In [2]:
# Базовое воспроизводимое окружение
from pathlib import Path
import json, os, random, time
import numpy as np

SEED = 42
FAST_DEV_RUN = False  # True — только проверка связности на уменьшенной выборке; не отчётный результат
# Отдельная подпапка: в artifacts/ лежат результаты лабораторной №1, в artifacts/lab2–lab5 — №2–№5.
ARTIFACTS = Path(os.getenv("CV_ARTIFACTS", "artifacts/lab6"))
ARTIFACTS.mkdir(parents=True, exist_ok=True)
# Изображения PixMo тоже храним внутри папки лабораторной (по умолчанию open_eval_data пишет в data/lab06)
LAB_IMAGES = ARTIFACTS / "images"
LAB_IMAGES.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    torch = None
    DEVICE = "cpu"
# Датасеты курса лежат в общем каталоге <корень курса>/data:
# notebook выполняется из labs/lab_XX_*/, поэтому корень курса — cwd.parent.parent.
# Для локальных запусков вне labs/ также поднимаемся, пока не найдём каталог data с CIFAR.
_CANDIDATES = [Path.cwd().parent.parent / "data", Path.cwd().parent / "data", Path.cwd() / "data"]
DATA_ROOT = Path(os.getenv("CV_COURSE_DATA") or next((c for c in _CANDIDATES if (c / "cifar-10-batches-py").exists()), _CANDIDATES[0]))
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print({"device": DEVICE, "fast_dev_run": FAST_DEV_RUN, "data_root": str(DATA_ROOT),
       "artifacts": str(ARTIFACTS), "lab_images": str(LAB_IMAGES)})


{'device': 'cuda', 'fast_dev_run': False, 'data_root': 'c:\\Users\\bogom\\OneDrive\\Документы\\PSY\\6 курс\\МО\\data', 'artifacts': 'artifacts\\lab6', 'lab_images': 'artifacts\\lab6\\images'}


In [3]:
def check(condition, message):
    """Мини-проверка в стиле YAAM: молчит, если всё хорошо; печатает, если найдена проблема."""
    if condition:
        print(f"✔ {message}")
    else:
        print(f"✘ ВНИМАНИЕ: {message}")

### Задание 1. Оценочный набор с происхождением

*Вес: 1 балл.*

**Постановка.** Используйте PixMo-Cap, PixMo-AskModelAnything, PixMo-Docs и PixMo-Count с опубликованными эталонами и сохраняйте происхождение каждой записи; локальных файлов от студента не требуется.

Выполните следующие действия:
1. Загрузите открытые изображения и опубликованные ответы через `open_eval_data.load_open_eval`; сохраните URL, split, индекс записи и сведения о лицензировании.
2. Проверьте размер набора и наличие captioning, VQA, OCR и structured extraction.

**Результат.** eval.jsonl с проверяемыми примерами и абсолютными либо существующими путями.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


**Задание 1. Оценочный набор с происхождением**

- **Что делали**: через `open_eval_data.load_open_eval` скачали 20 примеров PixMo (по 5 на captioning, VQA, OCR и подсчёт) с опубликованными эталонами. Картинки сохранили в `artifacts/lab6/images/`, у каждой записи сохранили источник, split, номер строки, лицензию и SHA256.
- **Зачем**: оценивать модель на реальных данных с проверяемыми эталонами, чтобы любой пример можно было найти в исходном датасете.
- **Результат**: eval.jsonl, 20 примеров, 4 задачи × 5. Все 8 проверок ✔: пути абсолютные и существуют, у каждой записи есть источник, лицензия и эталон.


In [4]:
# Задание 1. Оценочный набор с происхождением
from collections import Counter
from open_eval_data import load_open_eval

# Профиль вычислений: 5 примеров на каждый из четырёх типов задач; в FAST_DEV_RUN — по 1 (только smoke-проверка)
PER_TASK = 1 if FAST_DEV_RUN else 5
EXPECTED_TASKS = {"captioning", "vqa", "ocr", "structured"}
EVAL_PATH = ARTIFACTS / ("eval_fast.jsonl" if FAST_DEV_RUN else "eval.jsonl")

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

# 1. Повторный запуск не ходит в сеть, если eval.jsonl уже собран и все изображения на месте
cached = read_jsonl(EVAL_PATH) if EVAL_PATH.exists() else []
if len(cached) == PER_TASK * len(EXPECTED_TASKS) and all(Path(r["image"]).is_file() for r in cached):
    eval_rows = cached
    print(f"Используем сохранённый {EVAL_PATH}")
else:
    # Изображения скачиваются в artifacts/lab6/images, SHA256 сверяется с опубликованным (где он есть)
    raw = load_open_eval(root=LAB_IMAGES, per_task=PER_TASK)
    eval_rows = [{
        "id": f"{ex['task']}-{ex['source_row']}",
        **ex,
        # Для PixMo-Docs картинка лежит внутри датасета (постоянного URL нет) — происхождение задаётся source + split + row
        "provenance": f"{ex['source']} [split={ex['source_split']}, row={ex['source_row']}]",
    } for ex in raw]
    write_jsonl(EVAL_PATH, eval_rows)
    print(f"Сохранено {len(eval_rows)} примеров в {EVAL_PATH}")

# 2. Проверки размера, покрытия задач и происхождения
task_counts = Counter(r["task"] for r in eval_rows)
print("Примеров по задачам:", dict(task_counts))
check(len(eval_rows) == PER_TASK * len(EXPECTED_TASKS), f"размер набора {len(eval_rows)} = {PER_TASK} × {len(EXPECTED_TASKS)}")
check(set(task_counts) == EXPECTED_TASKS, "есть captioning, VQA, OCR и structured extraction")
check(all(n == PER_TASK for n in task_counts.values()), "задачи сбалансированы")
check(all(Path(r["image"]).is_absolute() and Path(r["image"]).is_file() for r in eval_rows), "все пути к изображениям абсолютные и существуют")
check(all(Path(r["image"]).resolve().is_relative_to(LAB_IMAGES.resolve()) for r in eval_rows), "изображения лежат в папке лабораторной")
check(all(r["source"] and r["source_split"] and r["source_row"] is not None and r["license"] for r in eval_rows),
      "у каждой записи есть источник, split, индекс и лицензия")
check(all(r["reference"] for r in eval_rows), "у каждой записи есть опубликованный эталон")
check(len({r["id"] for r in eval_rows}) == len(eval_rows), "идентификаторы уникальны")

# По одному примеру каждой задачи — для визуальной проверки
for task in sorted(EXPECTED_TASKS):
    r = next(x for x in eval_rows if x["task"] == task)
    print(f"\n[{task} / {r['category']}] {r['provenance']}\n  Q: {r['question'][:120]}\n  A: {str(r['reference'])[:160]}")


Используем сохранённый artifacts\lab6\eval.jsonl
Примеров по задачам: {'captioning': 5, 'vqa': 5, 'ocr': 5, 'structured': 5}
✔ размер набора 20 = 5 × 4
✔ есть captioning, VQA, OCR и structured extraction
✔ задачи сбалансированы
✔ все пути к изображениям абсолютные и существуют
✔ изображения лежат в папке лабораторной
✔ у каждой записи есть источник, split, индекс и лицензия
✔ у каждой записи есть опубликованный эталон
✔ идентификаторы уникальны

[captioning / caption] https://huggingface.co/datasets/allenai/pixmo-cap [split=train, row=0]
  Q: Describe the image.
  A: This photograph depicts a striking black bird, possibly a grackle or similar species, perched on a white cement wall with red stains. The bird's sleek, elongate

[ocr / document_qa] https://huggingface.co/datasets/allenai/pixmo-docs [split=validation, row=0]
  Q: What is the total estimated cost?
  A: $2,500

[structured / object_count_json] https://huggingface.co/datasets/allenai/pixmo-count [split=validation, row=12]
  Q

**Датасеты задания 1.** 20 реальных примеров PixMo от Allen AI, по 5 на каждую задачу.

| Задача | Датасет | Что на картинках | Что спрашивается | Эталон |
|---|---|---|---|---|
| **captioning** | PixMo-Cap | обычные фото и рисунки | «Describe the image.» | подробное описание |
| **vqa** | PixMo-AskModelAnything | объекты: тележка для попкорна, камера, самолёт | что это за объект | развёрнутый ответ |
| **ocr** | PixMo-Docs | документы: смета, SWOT-анализ, медкарта | найти значение в документе | короткая строка, например `$2,500` |
| **structured** | PixMo-Count | фото с людьми, самолётами, бутылками | сколько объектов, ответ в JSON | `{"label": "people", "count": 9}` |


### Задание 2. Модель и режим квантования

*Вес: 1 балл.*

**Постановка.** Фиксация версии модели и precision нужна для интерпретации результатов инференса.

Выполните следующие действия:
1. Загрузите processor и VLM.
2. На GPU включите NF4; без GPU не запускайте полный режим.

**Результат.** Объекты processor, model и model_id.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


VLM — нейросеть, которая принимает картинку с текстом и отвечает текстом. У нас это Qwen3-VL-4B-Instruct: около 4,4 млрд параметров.<br>
NF4 (NormalFloat 4) — способ сжать веса модели до 4 бит на параметр вместо обычных 16.

**Задание 2. Модель и режим квантования**

- **Что делали**: загрузили processor (переводит картинку и текст в числа модели, а ответ модели - обратно в текст. Выдает тензоры: номера токенов текста и пиксели патчей) и Qwen3-VL-4B-Instruct. Языковую часть сжали в NF4(каждый вес модели хранится в 4 битах, заместо 16) с двойным квантованием (сжатие служебных чисел), visual encoder (трансформер который смотрит на картинку) и `lm_head` (последний слой модели который раставляет вероятности для всех ~150 тысяч токенов словаря, а из них выбирается следующий токен ответа.) оставили в bf16 (их веса хранятся в обычном 16-битном формате, без потери точности). Записали ревизию весов, precision и память.
- **Зачем**: уместить модель 4B в видеокарту 12 ГБ и зафиксировать точную конфигурацию, без которой результаты нельзя интерпретировать и воспроизвести.
- **Результат**: model_config.json. Ревизия `ebb281ec`, 252 слоя `Linear4bit` в языковой модели и 0 в visual encoder, 3,19 ГБ памяти на RTX 3060, загрузка ≈ 24 с. Fallback 2B не понадобился.


In [5]:
# Задание 2. Модель и режим квантования
import subprocess, sys, importlib.util

# bitsandbytes (NF4) и accelerate (device_map) не входят в базовое окружение курса
_missing = [pkg for pkg in ("bitsandbytes", "accelerate") if importlib.util.find_spec(pkg) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

import transformers
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

# Основной режим — VLM 4B в NF4; fallback — 2B той же серии с тем же API (профиль вычислений)
PRIMARY_MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
FALLBACK_MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"

# 1. Без GPU полный режим не запускаем: NF4 работает только на CUDA, а fp32-инференс 4B на CPU не отражает отчётную конфигурацию
if DEVICE != "cuda" and not FAST_DEV_RUN:
    raise RuntimeError("GPU не найден: полный режим требует CUDA и NF4. Для smoke-проверки включите FAST_DEV_RUN=True.")

if DEVICE == "cuda":
    COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",           # 4-битный NormalFloat
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,  # матмулы считаются в bf16/fp16
        bnb_4bit_use_double_quant=True,       # квантуем и сами масштабы — ещё ~0.4 бит/параметр экономии
        # visual encoder и выходной слой оставляем в 16 битах. Шаблон сопоставляется с НАЧАЛОМ полного имени модуля
        # (re.match), поэтому нужен полный префикс "model.visual" — просто "visual" не срабатывает
        llm_int8_skip_modules=["model.visual", "lm_head"],
    )
else:  # только FAST_DEV_RUN на CPU: маленькая модель без квантования
    COMPUTE_DTYPE = torch.float32
    quant_config = None

def load_vlm(model_id):
    processor = AutoProcessor.from_pretrained(model_id)
    model = AutoModelForImageTextToText.from_pretrained(
        model_id,
        quantization_config=quant_config,
        dtype=COMPUTE_DTYPE,
        device_map="auto" if DEVICE == "cuda" else None,
    ).eval()
    return processor, model

# 2. Пробуем 4B; при нехватке памяти — fallback 2B. На CPU сразу 2B
candidates = [PRIMARY_MODEL_ID, FALLBACK_MODEL_ID] if DEVICE == "cuda" else [FALLBACK_MODEL_ID]
for model_id in candidates:
    try:
        t0 = time.perf_counter()
        processor, model = load_vlm(model_id)
        load_seconds = time.perf_counter() - t0
        break
    except torch.cuda.OutOfMemoryError as e:
        print(f"{model_id}: нехватка памяти GPU ({e}); переходим к fallback")
        torch.cuda.empty_cache()
else:
    raise RuntimeError("Не удалось загрузить ни основную, ни fallback-модель")

# Фиксируем точную ревизию весов (commit на Hugging Face Hub), чтобы результат можно было воспроизвести
try:
    from huggingface_hub import HfApi
    model_revision = HfApi().model_info(model_id).sha
except Exception:
    model_revision = getattr(model.config, "_commit_hash", None)

# Фактическая precision: считаем 4-битные слои внутри языковой модели и visual encoder
import bitsandbytes as bnb
n_4bit_lm = sum(isinstance(m, bnb.nn.Linear4bit) for name, m in model.named_modules() if "visual" not in name) if quant_config else 0
n_4bit_visual = sum(isinstance(m, bnb.nn.Linear4bit) for name, m in model.named_modules() if "visual" in name) if quant_config else 0
footprint_gb = model.get_memory_footprint() / 1024**3

model_card = {
    "model_id": model_id,
    "revision": model_revision,
    "is_fallback": model_id != PRIMARY_MODEL_ID,
    "n_params_billion": round(sum(p.numel() for p in model.parameters()) / 1e9, 3),
    "quantization": quant_config.to_dict() if quant_config else "none (CPU smoke, fp32)",
    "compute_dtype": str(COMPUTE_DTYPE),
    "linear4bit_layers": {"language_model": n_4bit_lm, "visual": n_4bit_visual},
    "memory_footprint_gb": round(footprint_gb, 2),
    "load_seconds": round(load_seconds, 1),
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
    "versions": {"torch": torch.__version__, "transformers": transformers.__version__,
                 "bitsandbytes": getattr(bnb, "__version__", None)},
    "fast_dev_run": FAST_DEV_RUN,
}
(ARTIFACTS / "model_config.json").write_text(json.dumps(model_card, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(model_card, ensure_ascii=False, indent=2))

check(processor is not None and model is not None, f"processor и model загружены: {model_id}")
check(model_revision is not None, f"ревизия весов зафиксирована: {model_revision}")
if DEVICE == "cuda":
    check(model_card["quantization"]["bnb_4bit_quant_type"] == "nf4", "режим квантования — NF4")
    check(n_4bit_lm > 0, f"языковая модель реально в 4 битах: {n_4bit_lm} слоёв Linear4bit")
    check(n_4bit_visual == 0, f"visual encoder не квантован (Linear4bit в visual: {n_4bit_visual})")
    check(footprint_gb < 6, f"модель занимает {footprint_gb:.2f} ГБ — помещается в бюджет GPU")


W1031 20:33:45.640000 9152 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

{
  "model_id": "Qwen/Qwen3-VL-4B-Instruct",
  "revision": "ebb281ec70b05090aa6165b016eac8ec08e71b17",
  "is_fallback": false,
  "n_params_billion": 2.621,
  "quantization": {
    "quant_method": "bitsandbytes",
    "_load_in_8bit": false,
    "_load_in_4bit": true,
    "llm_int8_threshold": 6.0,
    "llm_int8_skip_modules": [
      "model.visual",
      "lm_head"
    ],
    "llm_int8_enable_fp32_cpu_offload": false,
    "llm_int8_has_fp16_weight": false,
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_quant": true,
    "bnb_4bit_compute_dtype": "bfloat16",
    "bnb_4bit_quant_storage": "uint8",
    "load_in_4bit": true,
    "load_in_8bit": false
  },
  "compute_dtype": "torch.bfloat16",
  "linear4bit_layers": {
    "language_model": 252,
    "visual": 0
  },
  "memory_footprint_gb": 3.19,
  "load_seconds": 24.1,
  "device": "cuda",
  "gpu": "NVIDIA GeForce RTX 3060",
  "versions": {
    "torch": "2.11.0+cu128",
    "transformers": "5.17.0",
    "bitsandbytes": "0.50.2"
  },

### Задание 3. Chat template и функция инференса

*Вес: 1 балл.*

**Постановка.** Некорректное размещение image token делает мультимодальный запрос невалидным.

Выполните следующие действия:
1. Соберите сообщение с изображением и вопросом.
2. Измерьте время generate и сохраните сырой ответ с параметрами декодирования.

**Результат.** Функции build_prompt и run_vlm.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


**Задание 3. Chat template и функция инференса**

- **Что делали**: написали `build_prompt` (картинка перед вопросом в chat template модели) и `run_vlm` (уменьшение картинки (если она большая сжимается), generate (Метод модели, который пишет ответ: по одному токену, пока не выдаст <|im_end|> или не упрётся в max_new_tokens. Именно его время мы и меряем.) с синхронизацией CUDA (Видеокарта работает асинхронно: Python отдаёт ей команды и сразу идёт дальше, не дожидаясь результата. torch.cuda.synchronize() до и после generate заставляет дождаться окончания вычислений, иначе замер времени будет занижен.), сырой ответ (Текст ровно в том виде, в каком его выдала модель: со звёздочками Markdown, без обрезки пробелов и без исправлений. Убираются только служебные токены вроде <|im_end|>. Нормализуется ответ потом, при подсчёте метрик, и только в копии.) и все параметры). Задали 2 шаблона (формулировка запроса: plain — вопрос как есть; short_answer — вопрос плюс «ответь одним словом, числом или короткой фразой».), 2 разрешения (наибольший размер картинки: default — площадь не больше 1024×1024; low — не больше 448×448.) и 2 режима декодирования (как выбирается следующий токен: greedy — всегда самый вероятный (жадное декодирование), ответ при повторе одинаковый; sampling — случайно среди вероятных (temperature 0,7, top-p 0,8, top-k 20), ответы разнообразнее.).
- **Зачем**: собрать валидный мультимодальный запрос — ровно один image token (метка в тексте prompt, которая показывает модели, куда вставить картинку. Потом процессор разворачивает его в нужное число визуальных токенов) на правильном месте — и честно мерить время генерации.
- **Результат**: функции build_prompt и run_vlm, inference_config.json. Пробный запрос `vqa-0`: 120 визуальных токенов из 137 входных, ответ «popcorn machine» верный, около 18–19 с на 170 токенов ответа.


In [6]:
# Задание 3. Chat template и функция инференса
from PIL import Image

# Шаблоны пользовательского текста: {question} — вопрос из eval.jsonl. "plain" — основной, остальные — для абляции
PROMPT_TEMPLATES = {
    "plain": "{question}",
    "short_answer": "{question}\nAnswer with a single word, number or short phrase. Do not explain.",
}
# Разрешение: ограничение площади картинки в пикселях до препроцессинга (контролируем сами, а не дефолтом процессора)
RESOLUTIONS = {"low": 448 * 448, "default": 1024 * 1024}
# Режимы декодирования. Для greedy явно обнуляем sampling-параметры из generation_config модели
DECODING = {
    "greedy": dict(do_sample=False, temperature=None, top_p=None, top_k=None),
    "sampling": dict(do_sample=True, temperature=0.7, top_p=0.8, top_k=20),
}
# Описания и открытые ответы PixMo-AskModelAnything развёрнутые — при 128 токенах они обрываются на полуслове
MAX_NEW_TOKENS = {"captioning": 256, "vqa": 256}
DEFAULT_MAX_NEW_TOKENS = 128

IMAGE_TOKEN = processor.image_token  # для Qwen3-VL: <|image_pad|>
IMAGE_TOKEN_ID = processor.tokenizer.convert_tokens_to_ids(IMAGE_TOKEN)

def resize_to_budget(image, max_pixels):
    """Уменьшает изображение с сохранением пропорций, если площадь больше max_pixels; маленькие не растягивает."""
    w, h = image.size
    if max_pixels is None or w * h <= max_pixels:
        return image
    scale = (max_pixels / (w * h)) ** 0.5
    return image.resize((max(1, int(w * scale)), max(1, int(h * scale))), Image.BICUBIC)

def build_prompt(question, template="plain"):
    """1. Сообщение в формате chat template: сначала изображение, затем текст вопроса.

    Возвращает messages и строку prompt, в которой chat template сам расставил
    <|vision_start|><|image_pad|><|vision_end|> на месте картинки и добавил начало ответа ассистента.
    """
    user_text = PROMPT_TEMPLATES[template].format(question=question)
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": user_text}]}]
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    # Ровно один image token, и он стоит раньше вопроса — иначе запрос мультимодально невалиден
    assert prompt.count(IMAGE_TOKEN) == 1, f"ожидался один {IMAGE_TOKEN}, найдено {prompt.count(IMAGE_TOKEN)}"
    assert prompt.index(IMAGE_TOKEN) < prompt.index(user_text[:30]), "image token должен стоять перед вопросом"
    return messages, prompt, user_text

@torch.inference_mode()
def run_vlm(image_path, question, task=None, template="plain", resolution="default", decoding="greedy",
            max_new_tokens=None, seed=SEED):
    """2. Один запрос к VLM: сырой ответ + время generate + все параметры, которые на него влияют."""
    _, prompt, user_text = build_prompt(question, template)
    max_new_tokens = max_new_tokens or MAX_NEW_TOKENS.get(task, DEFAULT_MAX_NEW_TOKENS)
    gen_kwargs = dict(DECODING[decoding], max_new_tokens=max_new_tokens)

    t0 = time.perf_counter()
    with Image.open(image_path) as im:
        orig_size = im.size
        image = resize_to_budget(im.convert("RGB"), RESOLUTIONS[resolution])
    inputs = processor(text=[prompt], images=[image], return_tensors="pt").to(model.device)
    preprocess_s = time.perf_counter() - t0

    # Процессор разворачивает один image token в столько токенов, сколько патчей после merge — сверяем с сеткой
    n_image_tokens = int((inputs["input_ids"] == IMAGE_TOKEN_ID).sum())
    merge = getattr(processor.image_processor, "merge_size", 2)
    expected = int(inputs["image_grid_thw"].prod(-1).sum()) // merge**2
    assert n_image_tokens == expected, f"image tokens {n_image_tokens} != patches {expected}"

    torch.manual_seed(seed)  # sampling воспроизводим при фиксированном seed
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    out = model.generate(**inputs, **gen_kwargs)
    if DEVICE == "cuda":
        torch.cuda.synchronize()  # CUDA асинхронна: без синхронизации время generate занижено
    latency_s = time.perf_counter() - t0

    n_in = inputs["input_ids"].shape[1]
    new_tokens = out[0, n_in:]
    raw = processor.decode(new_tokens, skip_special_tokens=True)  # сырой ответ: без strip и постобработки
    return {
        "raw_output": raw,
        "latency_s": round(latency_s, 4),
        "preprocess_s": round(preprocess_s, 4),
        "n_input_tokens": int(n_in),
        "n_image_tokens": n_image_tokens,
        "n_output_tokens": int(new_tokens.numel()),
        "hit_max_new_tokens": int(new_tokens.numel()) >= max_new_tokens,
        "image_size_orig": list(orig_size),
        "image_size_used": list(image.size),
        "prompt_template": template,
        "user_text": user_text,
        "resolution": resolution,
        "max_pixels": RESOLUTIONS[resolution],
        "decoding": decoding,
        "generation_params": {k: v for k, v in gen_kwargs.items() if v is not None},
        "seed": seed,
        "peak_gpu_mem_gb": round(torch.cuda.max_memory_allocated() / 1024**3, 2) if DEVICE == "cuda" else None,
        "model_id": model_id,
        "revision": model_revision,
    }

# Конфигурация инференса — обязательный артефакт; дальнейшие задания ссылаются на эти же имена
inference_config = {
    "model_id": model_id, "revision": model_revision,
    "prompt_templates": PROMPT_TEMPLATES, "resolutions_max_pixels": RESOLUTIONS,
    "decoding": {k: {kk: vv for kk, vv in v.items() if vv is not None} for k, v in DECODING.items()},
    "max_new_tokens": {**MAX_NEW_TOKENS, "default": DEFAULT_MAX_NEW_TOKENS}, "seed": SEED,
}
(ARTIFACTS / "inference_config.json").write_text(json.dumps(inference_config, ensure_ascii=False, indent=2), encoding="utf-8")

# Smoke-проверка на одном реальном примере из eval.jsonl
ex = next(r for r in eval_rows if r["task"] == "vqa")
_, demo_prompt, _ = build_prompt(ex["question"])
print("Prompt после chat template:\n" + demo_prompt)
res = run_vlm(ex["image"], ex["question"], task=ex["task"])
print("\nОтвет модели:", res["raw_output"][:500])
print("Эталон:", str(ex["reference"])[:300])
print({k: res[k] for k in ("latency_s", "preprocess_s", "n_input_tokens", "n_image_tokens", "n_output_tokens",
                           "image_size_orig", "image_size_used", "generation_params", "peak_gpu_mem_gb")})

check(demo_prompt.count(IMAGE_TOKEN) == 1, "в prompt ровно один image token, перед вопросом")
check(res["n_image_tokens"] > 0, f"картинка развёрнута в {res['n_image_tokens']} визуальных токенов")
check(len(res["raw_output"].strip()) > 0, "модель вернула непустой ответ")
check(res["latency_s"] > 0 and "generation_params" in res, "время generate и параметры декодирования сохранены")


Prompt после chat template:
<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>What type of machine is this?<|im_end|>
<|im_start|>assistant


Ответ модели: This is a **popcorn machine**, specifically a **classic popcorn cart** or **popcorn machine on a cart**.

It features:

- A **glass front** to display the popcorn popping process.
- A **red and gold decorative design**, typical of vintage or retro-style machines.
- A **cart with wheels** for mobility, often used at fairs, carnivals, or events.
- A **popper mechanism** inside, where kernels are heated and popped.
- A **storage compartment** below for popcorn or supplies.

These machines are often
Эталон: This is a vintage-style popcorn cart. It features a red body, glass front, and gold lettering. The cart is on wheels, making it mobile and perfect for events like carnivals, fairs, or movie theaters. It combines functionality with a charming 1950s aesthetic, ideal for making and displaying fresh pop
{'latency_s': 20.0261, '

### Задание 4. Возобновляемые предсказания

*Вес: 1 балл.*

**Постановка.** Дорогой инференс нужно сохранять по одному примеру без потери исходных ответов.

Выполните следующие действия:
1. Запустите VLM для каждого примера.
2. Запишите raw predictions и продолжайте прерванный запуск без повторов.

**Результат.** predictions.jsonl и predictions.json с фактическими выходами VLM.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


**Задание 4. Возобновляемые предсказания**

- **Что делали**: прогнали модель по всем 20 примерам в конфигурации `plain|default|greedy`. Каждый ответ сразу дописывали на диск, при перезапуске готовые `id` пропускали.
- **Зачем**: не терять дорогой инференс (работа модели после получения картинки + вопроса) при сбое и не считать одно и то же дважды.
- **Результат**: predictions.jsonl и predictions.json, 20 ответов без повторов. Медиана 11,8 с, всего 4,7 мин генерации. 6 из 20 ответов (3 описания и 3 VQA) упёрлись в лимит 256 токенов.


In [7]:
# Задание 4. Возобновляемые предсказания
from datetime import datetime, timezone

# Основная конфигурация; варианты для абляции — в задании 8
MAIN_CONFIG = dict(template="plain", resolution="default", decoding="greedy")
CONFIG_NAME = "{template}|{resolution}|{decoding}".format(**MAIN_CONFIG)
PRED_JSONL = ARTIFACTS / ("predictions_fast.jsonl" if FAST_DEV_RUN else "predictions.jsonl")
PRED_JSON = PRED_JSONL.with_suffix(".json")

def read_jsonl_safe(path):
    """Читает JSONL, пропуская обрезанную строку — она появляется, если процесс убили посреди записи."""
    rows, broken = [], 0
    if path.exists():
        with open(path, encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError:
                    broken += 1
    if broken:
        print(f"⚠ {path.name}: пропущено {broken} повреждённых строк — эти примеры будут посчитаны заново")
    return rows

def append_jsonl(path, row):
    """Дописывает одну запись и сразу сбрасывает её на диск: после падения теряется максимум текущий пример."""
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
        f.flush()
        os.fsync(f.fileno())

def is_same_run(r):
    # Готовым считаем только ответ той же модели, ревизии и конфигурации — иначе это другой эксперимент
    return r.get("config") == CONFIG_NAME and r.get("model_id") == model_id and r.get("revision") == model_revision

# 2. Возобновление: всё, что уже записано, не пересчитываем
existing = read_jsonl_safe(PRED_JSONL)
done_ids = {r["id"] for r in existing if is_same_run(r)}
todo = [ex for ex in eval_rows if ex["id"] not in done_ids]
print(f"{PRED_JSONL.name}: готово {len(done_ids)}/{len(eval_rows)}, осталось {len(todo)} (конфигурация {CONFIG_NAME})")

# 1. Инференс по одному примеру с немедленной записью
failed = []
for i, ex in enumerate(todo, 1):
    try:
        res = run_vlm(ex["image"], ex["question"], task=ex["task"], **MAIN_CONFIG)
    except torch.cuda.OutOfMemoryError as e:
        # Ошибку не записываем как предсказание: при следующем запуске пример будет повторён
        torch.cuda.empty_cache()
        failed.append(ex["id"]); print(f"  ✘ {ex['id']}: OOM ({e})"); continue
    record = {
        "id": ex["id"], "task": ex["task"], "category": ex["category"],
        "question": ex["question"], "reference": ex["reference"], "image": ex["image"],
        "config": CONFIG_NAME, **res,
        "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    append_jsonl(PRED_JSONL, record)
    print(f"  [{i}/{len(todo)}] {ex['id']:<16} {res['latency_s']:6.1f} с, {res['n_output_tokens']:3d} ток."
          f"{' (лимит)' if res['hit_max_new_tokens'] else ''} → {res['raw_output'][:70]!r}")

# Снимок в predictions.json: по одной записи на пример в порядке eval.jsonl (дубликаты исключены)
saved = [r for r in read_jsonl_safe(PRED_JSONL) if is_same_run(r)]
by_id = {r["id"]: r for r in saved}
predictions = [by_id[ex["id"]] for ex in eval_rows if ex["id"] in by_id]
PRED_JSON.write_text(json.dumps(predictions, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"\nСохранено: {PRED_JSONL} и {PRED_JSON} — {len(predictions)} предсказаний")
print("По задачам:", dict(Counter(p["task"] for p in predictions)))
print(f"Задержка generate: медиана {np.median([p['latency_s'] for p in predictions]):.1f} с, "
      f"сумма {sum(p['latency_s'] for p in predictions) / 60:.1f} мин")

check(len(predictions) == len(eval_rows), f"предсказания есть для всех {len(eval_rows)} примеров" + (f"; не удались: {failed}" if failed else ""))
check(len(saved) == len(predictions), "в predictions.jsonl нет повторов одного примера")
check(all(isinstance(p["raw_output"], str) and p["raw_output"].strip() for p in predictions), "все ответы — непустые сырые выходы VLM")
check(all(p["model_id"] == model_id and p["revision"] == model_revision for p in predictions), "все ответы получены одной моделью и ревизией")
n_cut = sum(p["hit_max_new_tokens"] for p in predictions)
check(n_cut == 0, f"ни один ответ не упёрся в max_new_tokens (упёрлись: {n_cut})")


predictions.jsonl: готово 20/20, осталось 0 (конфигурация plain|default|greedy)

Сохранено: artifacts\lab6\predictions.jsonl и artifacts\lab6\predictions.json — 20 предсказаний
По задачам: {'captioning': 5, 'vqa': 5, 'ocr': 5, 'structured': 5}
Задержка generate: медиана 11.8 с, сумма 4.7 мин
✔ предсказания есть для всех 20 примеров
✔ в predictions.jsonl нет повторов одного примера
✔ все ответы — непустые сырые выходы VLM
✔ все ответы получены одной моделью и ревизией
✘ ВНИМАНИЕ: ни один ответ не упёрся в max_new_tokens (упёрлись: 6)


**Что происходит в задании 4.** Модель **Qwen3-VL-4B-Instruct (NF4)** через `run_vlm` отвечает на все 20 примеров из `eval.jsonl` в базовой конфигурации `plain|default|greedy` (вопрос как есть, картинка ≤ 1024², жадное декодирование). Каждый ответ сразу дописывается в `predictions.jsonl`; при повторном запуске готовые `id` пропускаются, поэтому прерванный прогон продолжается без повторов. Итоговый снимок — `predictions.json`.

| Задача (`id`) | Датасет | Что спрашивается | Эталон |
|---|---|---|---|
| `captioning-*` | PixMo-Cap | «Describe the image.» — описать фото | подробное описание от человека |
| `vqa-*` | PixMo-AskModelAnything | что за объект на фото (машина, камера, транспорт, самолёт, модель) | развёрнутый ответ человека |
| `ocr-*` | PixMo-Docs | найти ответ в документе (сумма, возраст, бюджет, пункт SWOT) | короткая строка: `$2,500`, `30` |
| `structured-*` | PixMo-Count | сколько объектов (люди, самолёты, бутылки) — ответ строго в JSON | `{"label": ..., "count": ...}` |

**Как читать вывод.** Строка `[i/N] id время, токены → 'начало ответа'`: по префиксу `id` видно задачу и датасет; пометка `(лимит)` — ответ упёрся в `max_new_tokens`. Короткие ответы (OCR, подсчёт) занимают 1–3 с, описания и открытые VQA — 15–30 с. Номер в `id` — индекс строки в исходном датасете PixMo.


### Задание 5. Точная оценка коротких ответов

*Вес: 1 балл.*

**Постановка.** Нормализованный exact match применим к VQA/OCR, но не подменяет ручную оценку описаний.

Выполните следующие действия:
1. Нормализуйте предсказания и эталоны для VQA/OCR.
2. Измерьте точность по категориям.

**Результат.** metrics.json с реально измеренным exact match.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


**Задание 5. Точная оценка коротких ответов**

- **Что делали**: нормализовали ответы и эталоны (Markdown, регистр, пунктуация, артикли, разделители тысяч) и посчитали exact match, contains и token F1 для VQA и OCR. Для подсчёта разобрали JSON и сравнили число.
- **Зачем**: измерить точность коротких ответов автоматически. Описания исключили — для них только ручная оценка.
- **Результат**: metrics.json.
  - OCR: EM 0,00, contains 0,80, F1 0,31 — модель знает ответ, но пишет его полным предложением;
  - open VQA: EM 0,00, F1 0,31 — эталоны там длинные абзацы;
  - подсчёт: JSON валиден 5/5 (формат совпадает), число точно 4/5, средняя ошибка 0,2.


In [8]:
# Задание 5. Точная оценка коротких ответов
import re, string

ARTICLES = {"a", "an", "the"}
_PUNCT = str.maketrans({c: " " for c in string.punctuation + "“”‘’«»—–…"})

def normalize_answer(text):
    """1. Нормализация в духе VQA/SQuAD: Markdown, регистр, пунктуация, артикли, разделители тысяч, пробелы."""
    s = str(text)
    s = re.sub(r"```(?:json)?", " ", s)                        # блоки кода
    s = re.sub(r"[*_`#]+", " ", s)                              # **жирный**, `код`, # заголовки
    s = re.sub(r"^\s*[-•]\s+", " ", s, flags=re.M)              # маркеры списков
    s = s.lower()
    s = re.sub(r"(?<=\d)[,\s](?=\d{3}\b)", "", s)               # 2,500 → 2500
    s = re.sub(r"(?<=\d)\.0+\b", "", s)                         # 30.0 → 30
    s = s.translate(_PUNCT)
    tokens = [t for t in s.split() if t not in ARTICLES]
    return " ".join(tokens)

def exact_match(pred, ref):
    return float(normalize_answer(pred) == normalize_answer(ref))

def contains_match(pred, ref):
    """Ослабленный критерий (не EM): нормализованный эталон целиком входит в ответ как отдельные слова."""
    p, r = normalize_answer(pred), normalize_answer(ref)
    return float(bool(r) and re.search(rf"(?<!\S){re.escape(r)}(?!\S)", p) is not None)

def token_f1(pred, ref):
    """Пересечение мешков слов (SQuAD F1) — частичное совпадение для длинных ответов."""
    p, r = normalize_answer(pred).split(), normalize_answer(ref).split()
    common = sum((Counter(p) & Counter(r)).values())
    if not p or not r or common == 0:
        return 0.0
    prec, rec = common / len(p), common / len(r)
    return 2 * prec * rec / (prec + rec)

def parse_count_json(text):
    """Достаёт первый JSON-объект из ответа (модель может обернуть его в ```json ... ```)."""
    m = re.search(r"\{.*?\}", str(text), flags=re.S)
    if not m:
        return None
    try:
        obj = json.loads(m.group(0))
        return obj if isinstance(obj, dict) else None
    except json.JSONDecodeError:
        return None

def score_prediction(p):
    """Метрики одного ответа; captioning не оценивается автоматически — только ручная разметка."""
    if p["task"] in ("vqa", "ocr"):
        return {"exact_match": exact_match(p["raw_output"], p["reference"]),
                "contains_match": contains_match(p["raw_output"], p["reference"]),
                "token_f1": round(token_f1(p["raw_output"], p["reference"]), 4)}
    if p["task"] == "structured":
        ref, got = json.loads(p["reference"]), parse_count_json(p["raw_output"])
        valid = got is not None and {"label", "count"} <= set(got)
        cnt = got.get("count") if valid else None
        cnt = int(cnt) if isinstance(cnt, (int, float)) or (isinstance(cnt, str) and cnt.strip().isdigit()) else None
        return {"json_valid": float(valid),
                "label_match": float(valid and normalize_answer(got["label"]) == normalize_answer(ref["label"])),
                "count_exact": float(cnt is not None and cnt == ref["count"]),
                "count_abs_error": abs(cnt - ref["count"]) if cnt is not None else None}
    return {}

def aggregate(rows):
    """Средние по метрикам группы; None (например, неразобранный count) в среднее не входит."""
    keys = sorted({k for r in rows for k in r["scores"]})
    out = {"n": len(rows)}
    for k in keys:
        vals = [r["scores"][k] for r in rows if r["scores"].get(k) is not None]
        out[k] = round(float(np.mean(vals)), 4) if vals else None
        if k == "count_abs_error":
            out["count_n_parsed"] = len(vals)
    return out

# Берём сохранённые предсказания с диска — так ячейка работает и после перезапуска ядра
predictions = json.loads(PRED_JSON.read_text(encoding="utf-8"))
scored = [{**{k: p[k] for k in ("id", "task", "category", "question", "reference", "raw_output")},
           "pred_normalized": normalize_answer(p["raw_output"]) if p["task"] in ("vqa", "ocr") else None,
           "ref_normalized": normalize_answer(p["reference"]) if p["task"] in ("vqa", "ocr") else None,
           "scores": score_prediction(p)} for p in predictions]

# 2. Точность по категориям
by_cat = {}
for r in scored:
    by_cat.setdefault(r["category"], []).append(r)
per_category = {cat: aggregate(rows) for cat, rows in by_cat.items() if cat != "caption"}
short = [r for r in scored if r["task"] in ("vqa", "ocr")]

metrics = {
    "model_id": model_id, "revision": model_revision, "config": CONFIG_NAME,
    "n_predictions": len(scored), "n_images": len({p["image"] for p in predictions}),
    "short_answers_vqa_ocr": aggregate(short),
    "per_category": per_category,
    "caption": {"n": len(by_cat.get("caption", [])),
                "note": "описания не оцениваются exact match — только ручная разметка (manual_review.jsonl)"},
    "definitions": {
        "exact_match": "normalize(pred) == normalize(ref); нормализация: Markdown, регистр, пунктуация, артикли a/an/the, разделители тысяч",
        "contains_match": "ослабленный критерий: нормализованный эталон входит в ответ целиком; не является exact match",
        "token_f1": "F1 по пересечению слов нормализованных строк (SQuAD)",
        "structured": "JSON разбирается из ответа; count_exact — точное совпадение числа, count_abs_error — |pred − ref|",
    },
    "per_example": [{k: r[k] for k in ("id", "category", "pred_normalized", "ref_normalized", "scores")} for r in scored],
    "fast_dev_run": FAST_DEV_RUN,
}
(ARTIFACTS / "metrics.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Конфигурация {CONFIG_NAME}, предсказаний: {len(scored)}\n")
print(f"{'категория':<20}{'n':>3}{'EM':>8}{'contains':>10}{'F1':>8}")
for cat in ("document_qa", "open_vqa"):
    if cat in per_category:
        m = per_category[cat]
        print(f"{cat:<20}{m['n']:>3}{m['exact_match']:>8.2f}{m['contains_match']:>10.2f}{m['token_f1']:>8.2f}")
m = metrics["short_answers_vqa_ocr"]
print(f"{'VQA+OCR вместе':<20}{m['n']:>3}{m['exact_match']:>8.2f}{m['contains_match']:>10.2f}{m['token_f1']:>8.2f}")
if "object_count_json" in per_category:
    m = per_category["object_count_json"]
    print(f"\nobject_count_json (n={m['n']}): JSON валиден {m['json_valid']:.2f}, метка {m['label_match']:.2f}, "
          f"count точно {m['count_exact']:.2f}, средняя |ошибка| {m['count_abs_error']}")

print("\nПримеры нормализации (VQA/OCR):")
for r in short:
    print(f"  {r['id']:<8} EM={r['scores']['exact_match']:.0f} contains={r['scores']['contains_match']:.0f} "
          f"| ref: {r['ref_normalized'][:35]!r} | pred: {r['pred_normalized'][:60]!r}")

check(len(short) > 0, f"exact match посчитан на {len(short)} реальных ответах VQA/OCR")
check(all(cat in per_category for cat in ("document_qa", "open_vqa", "object_count_json")), "точность посчитана по всем категориям коротких ответов")
check("caption" not in per_category, "описания исключены из exact match — для них ручная оценка")
check((ARTIFACTS / "metrics.json").exists(), f"metrics.json сохранён в {ARTIFACTS}")


Конфигурация plain|default|greedy, предсказаний: 20

категория             n      EM  contains      F1
document_qa           5    0.00      0.80    0.31
open_vqa              5    0.00      0.00    0.31
VQA+OCR вместе       10    0.00      0.40    0.31

object_count_json (n=5): JSON валиден 1.00, метка 1.00, count точно 0.80, средняя |ошибка| 0.2

Примеры нормализации (VQA/OCR):
  vqa-0    EM=0 contains=0 | ref: 'this is vintage style popcorn cart ' | pred: 'this is popcorn machine specifically classic popcorn cart or'
  vqa-1    EM=0 contains=0 | ref: 'this is polaroid spectra system se ' | pred: 'this is polaroid spectra system se instant camera key featur'
  vqa-2    EM=0 contains=0 | ref: 'this vehicle is paratrike it s uniq' | pred: 'this is motorized hang glider also known as motorized paragl'
  vqa-3    EM=0 contains=0 | ref: 'aircraft in this image is nasa supe' | pred: 'aircraft in image is nasa dc 9 modified boeing 707 used for '
  vqa-5    EM=0 contains=0 | ref: 'this image 

**Метрики задания 5.** Перед сравнением ответ модели и эталон нормализуются: удаляется Markdown (`**`, `#`, списки), нижний регистр, без пунктуации и артиклей a/an/the, `$2,500` → `2500`. Сырой ответ в `predictions.jsonl` не меняется.

| Метрика | Как считается | Пример | Что показывает |
|---|---|---|---|
| **EM** (exact match) | нормализованные строки совпадают целиком → 1, иначе 0 | `2500` vs `total estimated cost is 2500` → **0** | строгая точность; основная метрика задания |
| **contains** | нормализованный эталон целиком входит в ответ. 1, если ref целиком входит в pred; иначе 0 | `2500` внутри `… cost is 2500` → **1** | ответ верный, но многословный; ослабленный критерий, **не EM** |
| **token F1** | F1 по общим словам: P = общие / слова ответа, R = общие / слова эталона, F1 = 2PR/(P+R) | эталон 4 слова, ответ 10, общих 3 → P = 0.3, R = 0.75, **F1 ≈ 0.43** | частичное совпадение для длинных ответов (как в SQuAD) |

**Подсчёт объектов (structured):** `json_valid` — ответ разбирается как JSON с ключами `label` и `count`; `label_match` — совпала метка; `count_exact` — число совпало точно; `count_abs_error` — |ответ − эталон|.

**Как читать результат.** Разрыв между низким EM и высоким contains значит, что модель знает ответ, но отвечает предложением (шаблон `plain`), — это проверяет абляция с `short_answer`. Для `open_vqa` эталоны — длинные абзацы, поэтому EM там почти всегда 0 и неинформативен; описания (`caption`) вообще не оцениваются автоматически — для них ручная разметка.

pred написала модель, а ref — правильный ответ из датасета.


### Задание 6. Заготовка экспертной разметки

*Вес: 1 балл.*

**Постановка.** Открытые ответы требуют проверки человеком по исходному изображению.

Выполните следующие действия:
1. Сформируйте отдельные поля correctness и типологии ошибок.
2. Не перезаписывайте ранее заполненную разметку.

**Результат.** manual_review.jsonl с исходными ответами и полями экспертной оценки.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


**Задание 6. Заготовка экспертной разметки**

- **Что делали**: для каждого ответа создали строку с картинкой, вопросом, эталоном, сырым ответом и пустыми полями оценки: correctness, 6 типов ошибок, число утверждений и неподтверждённых, комментарий. Записывали атомарно, уже заполненные строки не трогали.
- **Зачем**: подготовить ручную проверку открытых ответов по изображению, не рискуя потерять сделанную разметку.
- **Результат**: manual_review.jsonl, 20 строк. При повторном запуске добавлено 0, заполненная разметка не изменилась.


In [15]:
# Задание 6. Заготовка экспертной разметки
REVIEW_PATH = ARTIFACTS / ("manual_review_fast.jsonl" if FAST_DEV_RUN else "manual_review.jsonl")

# 1. Схема экспертной оценки: correctness и типология ошибок — отдельные поля, заполняются человеком по изображению
CORRECTNESS_VALUES = ["correct", "partially_correct", "incorrect"]
ERROR_TYPES = {
    "hallucination": "утверждение о том, чего нет на изображении",
    "ocr": "неверно прочитан или не найден текст/число",
    "spatial": "ошибка в расположении объектов (слева/справа, над/под)",
    "counting": "неверное число объектов",
    "wrong_object": "объект распознан неверно (не тот тип/модель)",
    "format": "нарушен требуемый формат ответа (например, не JSON)",
}
REVIEW_FIELDS = ("correctness", "errors", "n_claims", "n_unsupported_claims", "comment", "reviewer", "reviewed_at")

def empty_review():
    return {
        "correctness": None,                              # одно из CORRECTNESS_VALUES
        "errors": {t: None for t in ERROR_TYPES},         # True / False по каждому типу ошибки
        "n_claims": None,                                 # число проверяемых утверждений в ответе
        "n_unsupported_claims": None,                     # из них не подтверждены изображением → H = n_unsupported / n_claims
        "comment": "",
        "reviewer": None,
        "reviewed_at": None,
    }

def is_reviewed(row):
    return row.get("correctness") is not None

def review_key(row):
    return (row["id"], row["config"])

# 2. Не перезаписываем разметку: существующие строки сохраняются целиком, добавляются только новые примеры
existing_reviews = read_jsonl_safe(REVIEW_PATH)
existing_by_key = {review_key(r): r for r in existing_reviews}
filled_before = {review_key(r): json.dumps(r, sort_keys=True, ensure_ascii=False) for r in existing_reviews if is_reviewed(r)}

predictions = json.loads(PRED_JSON.read_text(encoding="utf-8"))
auto_scores = {r["id"]: r["scores"] for r in scored} if "scored" in globals() else {}
added = 0
for p in predictions:
    key = (p["id"], p["config"])
    if key in existing_by_key:
        continue
    existing_by_key[key] = {
        # Исходные данные для проверки — ответ модели не меняется
        "id": p["id"], "task": p["task"], "category": p["category"], "config": p["config"],
        "image": p["image"], "question": p["question"], "reference": p["reference"], "raw_output": p["raw_output"],
        "model_id": p["model_id"], "revision": p["revision"],
        "auto_scores": auto_scores.get(p["id"], {}),  # подсказка из задания 5, не замена ручной оценки
        **empty_review(),
    }
    added += 1

# Порядок: сначала текущие предсказания в порядке eval.jsonl, затем строки прошлых конфигураций (если были)
order = {(p["id"], p["config"]): i for i, p in enumerate(predictions)}
reviews = sorted(existing_by_key.values(), key=lambda r: order.get(review_key(r), len(order)))

# Атомарная запись: сначала во временный файл, потом замена — при сбое старая разметка не теряется
tmp_path = REVIEW_PATH.with_suffix(".jsonl.tmp")
write_jsonl(tmp_path, reviews)
os.replace(tmp_path, REVIEW_PATH)

n_done = sum(is_reviewed(r) for r in reviews)
print(f"{REVIEW_PATH}: всего {len(reviews)} строк, добавлено {added}, уже размечено {n_done}, ожидают оценки {len(reviews) - n_done}")
print("По задачам:", dict(Counter(r["task"] for r in reviews)))
print("Поля экспертной оценки:", ", ".join(REVIEW_FIELDS))
print("Типы ошибок:", ", ".join(ERROR_TYPES))

reloaded = read_jsonl_safe(REVIEW_PATH)
reloaded_by_key = {review_key(r): r for r in reloaded}
check(all(key in reloaded_by_key for key in order), f"в разметке есть строки для всех {len(predictions)} предсказаний")
check(all(json.dumps(reloaded_by_key[k], sort_keys=True, ensure_ascii=False) == v for k, v in filled_before.items()),
      f"ранее заполненная разметка не изменена ({len(filled_before)} строк)")
check(all(set(REVIEW_FIELDS) <= set(r) and set(r["errors"]) == set(ERROR_TYPES) for r in reloaded),
      "у каждой строки есть поля correctness, типология ошибок и счётчики утверждений")
check(all(r["raw_output"] == next(p["raw_output"] for p in predictions if p["id"] == r["id"])
          for r in reloaded if review_key(r) in order), "исходные ответы модели перенесены без изменений")


artifacts\lab6\manual_review.jsonl: всего 20 строк, добавлено 0, уже размечено 20, ожидают оценки 0
По задачам: {'captioning': 5, 'vqa': 5, 'ocr': 5, 'structured': 5}
Поля экспертной оценки: correctness, errors, n_claims, n_unsupported_claims, comment, reviewer, reviewed_at
Типы ошибок: hallucination, ocr, spatial, counting, wrong_object, format
✔ в разметке есть строки для всех 20 предсказаний
✔ ранее заполненная разметка не изменена (20 строк)
✔ у каждой строки есть поля correctness, типология ошибок и счётчики утверждений
✔ исходные ответы модели перенесены без изменений


**Поля экспертной оценки** — то, что человек заполняет по каждому ответу модели:

| Поле | Что означает |
|---|---|
| `correctness` | общий вердикт: `correct`, `partially_correct` или `incorrect` |
| `errors` | какие типы ошибок есть в ответе, да/нет по каждому |
| `n_claims` | сколько проверяемых утверждений в ответе |
| `n_unsupported_claims` | сколько из них картинка не подтверждает |
| `comment` | что именно не так, своими словами |
| `reviewer` | кто проверял |
| `reviewed_at` | когда проверили, заполняется автоматически |

**Типы ошибок:**

| Тип | Что означает | Пример |
|---|---|---|
| `hallucination` | утверждение о том, чего на картинке нет | бортовой номер N707NA |
| `ocr` | неверно прочитан текст или число | «25 мм» вместо «125 мм» |
| `spatial` | перепутано расположение | ключи на руле, а не в замке зажигания |
| `counting` | неверное количество | 8 людей вместо 9 |
| `wrong_object` | объект назван неправильно | Hummer вместо гаубицы G6 |
| `format` | нарушен требуемый формат | просили JSON, а пришёл текст |


### Задание 7. Ручная типология ошибок

*Вес: 1 балл.*

**Постановка.** Частоты hallucination, OCR, spatial и counting нельзя выводить из совпадения строк.

Выполните следующие действия:
1. Заполните ручные поля по изображению и эталону.
2. Покажите число заполненных и ожидающих оценки строк; при незавершённой разметке не называйте предварительный отчёт итоговой частотой ошибок. Форма создаётся внутри ноутбука.

**Результат.** error_report.json с числом реально просмотренных примеров.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


**Задание 7. Ручная типология ошибок**

- **Что делали**: в форме внутри ноутбука вручную оценили все 20 ответов по картинке и эталону. Отчёт пересобирался после каждого сохранения и, пока были неоценённые строки, помечался как предварительный.
- **Зачем**: частоты галлюцинаций, ошибок OCR, расположения и подсчёта нельзя получить сравнением строк.
- **Результат**: error_report.json, просмотрено 20/20, статус final.
  - correctness: 11 correct, 6 partially_correct, 3 incorrect;
  - ошибки: hallucination 4, wrong_object 4, ocr 2, spatial 1, format 0;
  - доля неподтверждённых утверждений H = 24 (сколько из них картинка не подтверждает: неверные или непроверяемые.) / 130 (проверяемых фактов в ответе(«птица чёрная», «смотрит вправо», «на объективе 25 мм»…).) ≈ 0,18 (Примерно каждое пятое утверждение модели не подтверждается картинкой. Оставшиеся 82% (106 из 130) — верные факты.).


In [10]:
# Задание 7. Ручная типология ошибок
import io
from datetime import datetime, timezone
import ipywidgets as W
from IPython.display import display

ERROR_REPORT_PATH = ARTIFACTS / ("error_report_fast.json" if FAST_DEV_RUN else "error_report.json")

def load_reviews():
    return read_jsonl_safe(REVIEW_PATH)

def save_reviews(rows):
    tmp = REVIEW_PATH.with_suffix(".jsonl.tmp")  # атомарная запись, как в задании 6
    write_jsonl(tmp, rows)
    os.replace(tmp, REVIEW_PATH)

def set_review(row_id, correctness, errors, n_claims=None, n_unsupported_claims=None, comment="", reviewer=None,
               config=CONFIG_NAME):
    """Записывает экспертную оценку одной строки. Меняются только поля оценки — ответ модели и эталон не трогаем."""
    assert correctness in CORRECTNESS_VALUES, f"correctness должен быть одним из {CORRECTNESS_VALUES}"
    assert set(errors) <= set(ERROR_TYPES), f"неизвестные типы ошибок: {set(errors) - set(ERROR_TYPES)}"
    if n_claims is not None and n_unsupported_claims is not None:
        assert 0 <= n_unsupported_claims <= n_claims, "неподтверждённых утверждений не может быть больше, чем всех"
    rows = load_reviews()
    row = next(r for r in rows if r["id"] == row_id and r["config"] == config)
    row.update(correctness=correctness,
               errors={t: bool(errors.get(t, False)) for t in ERROR_TYPES},
               n_claims=n_claims, n_unsupported_claims=n_unsupported_claims,
               comment=comment, reviewer=reviewer or None,
               reviewed_at=datetime.now(timezone.utc).isoformat(timespec="seconds"))
    save_reviews(rows)
    return row

def build_error_report(config=CONFIG_NAME):
    """Отчёт строится только по реально просмотренным строкам; пока есть ожидающие — он помечен как предварительный."""
    rows = [r for r in load_reviews() if r["config"] == config]
    done = [r for r in rows if is_reviewed(r)]
    pending = [r["id"] for r in rows if not is_reviewed(r)]
    is_final = len(done) > 0 and not pending

    def freq(subset):
        n = len(subset)
        return {"n_reviewed": n,
                "correctness": {c: sum(r["correctness"] == c for r in subset) for c in CORRECTNESS_VALUES},
                "error_counts": {t: sum(bool(r["errors"].get(t)) for r in subset) for t in ERROR_TYPES},
                "error_rates": {t: round(sum(bool(r["errors"].get(t)) for r in subset) / n, 4) if n else None
                                for t in ERROR_TYPES}}

    with_claims = [r for r in done if r.get("n_claims")]
    n_claims = sum(r["n_claims"] for r in with_claims)
    n_unsup = sum(r["n_unsupported_claims"] or 0 for r in with_claims)
    report = {
        "status": "final" if is_final else "preliminary",
        "note": ("все строки просмотрены — частоты итоговые" if is_final else
                 f"ПРЕДВАРИТЕЛЬНЫЙ отчёт: просмотрено {len(done)} из {len(rows)}; частоты не являются итоговыми"),
        "config": config, "model_id": model_id, "revision": model_revision,
        "n_total": len(rows), "n_reviewed": len(done), "n_pending": len(pending), "pending_ids": pending,
        "overall": freq(done),
        "per_task": {t: freq([r for r in done if r["task"] == t]) for t in sorted({r["task"] for r in rows})},
        "hallucination_rate_claims": {
            "H": round(n_unsup / n_claims, 4) if n_claims else None,
            "n_unsupported_claims": n_unsup, "n_claims": n_claims, "n_examples_with_claims": len(with_claims),
            "definition": "H = N_unsupported / N_claims по просмотренным примерам с подсчитанными утверждениями",
        },
        "error_type_definitions": ERROR_TYPES,
        "reviewed_examples": [{"id": r["id"], "task": r["task"], "correctness": r["correctness"],
                               "errors": [t for t, v in r["errors"].items() if v], "comment": r["comment"]} for r in done],
        "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    ERROR_REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    return report

def image_bytes(path, max_side=900):
    with Image.open(path) as im:
        im = im.convert("RGB")
        im.thumbnail((max_side, max_side))
        buf = io.BytesIO(); im.save(buf, format="JPEG", quality=90)
        return buf.getvalue()

# 1. Форма экспертной разметки внутри ноутбука: изображение + вопрос + эталон + ответ модели → ручные поля
def review_form(config=CONFIG_NAME):
    ids = [r["id"] for r in load_reviews() if r["config"] == config]
    state = {"i": 0}

    img = W.Image(format="jpeg", layout=W.Layout(max_width="560px", max_height="560px", object_fit="contain"))
    info = W.HTML(layout=W.Layout(width="100%"))
    correctness = W.ToggleButtons(options=[("— не оценено —", None)] + [(c, c) for c in CORRECTNESS_VALUES],
                                  description="correctness")
    err_boxes = {t: W.Checkbox(description=f"{t}", tooltip=d, indent=False, layout=W.Layout(width="150px"))
                 for t, d in ERROR_TYPES.items()}
    n_claims = W.BoundedIntText(min=0, max=100, description="утверждений", layout=W.Layout(width="200px"))
    n_unsup = W.BoundedIntText(min=0, max=100, description="неподтв.", layout=W.Layout(width="200px"))
    comment = W.Textarea(placeholder="что именно не так (по изображению)", layout=W.Layout(width="95%", height="60px"))
    reviewer = W.Text(description="эксперт", value="", layout=W.Layout(width="300px"))
    status = W.HTML()
    b_prev, b_next = W.Button(description="◀ назад"), W.Button(description="вперёд ▶")
    b_save = W.Button(description="Сохранить", button_style="success", icon="check")
    b_pending = W.Button(description="к следующей неоценённой", icon="forward")

    def esc(s):
        return str(s).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")

    def show():
        rows = {r["id"]: r for r in load_reviews() if r["config"] == config}
        r = rows[ids[state["i"]]]
        img.value = image_bytes(r["image"])
        info.value = (f"<b>{state['i'] + 1}/{len(ids)} · {r['id']}</b> ({r['task']} / {r['category']})<br>"
                      f"<b>Вопрос:</b> {esc(r['question'])}<br><b>Эталон:</b> {esc(r['reference'])}<br>"
                      f"<b>Ответ модели:</b><pre style='white-space:pre-wrap;max-height:260px;overflow:auto'>"
                      f"{esc(r['raw_output'])}</pre><b>Авто-метрики (подсказка):</b> {r.get('auto_scores')}")
        correctness.value = r["correctness"]
        for t, cb in err_boxes.items():
            cb.value = bool(r["errors"].get(t))
        n_claims.value = r["n_claims"] or 0
        n_unsup.value = r["n_unsupported_claims"] or 0
        comment.value = r["comment"] or ""
        reviewer.value = r["reviewer"] or reviewer.value
        n_done = sum(is_reviewed(x) for x in rows.values())
        status.value = (f"Строка: <b>{'оценена' if is_reviewed(r) else 'ожидает оценки'}</b> · "
                        f"всего оценено {n_done}/{len(rows)}, ожидают {len(rows) - n_done}")

    def on_save(_):
        if correctness.value is None:
            status.value = "<span style='color:#c00'>Выберите correctness перед сохранением</span>"; return
        if n_unsup.value > n_claims.value:
            status.value = "<span style='color:#c00'>Неподтверждённых утверждений больше, чем всех</span>"; return
        set_review(ids[state["i"]], correctness.value, {t: cb.value for t, cb in err_boxes.items()},
                   n_claims=n_claims.value or None, n_unsupported_claims=n_unsup.value if n_claims.value else None,
                   comment=comment.value, reviewer=reviewer.value, config=config)
        rep = build_error_report(config)
        show()
        status.value += f" · сохранено, error_report.json обновлён ({rep['status']})"

    def move(step):
        state["i"] = (state["i"] + step) % len(ids); show()

    def to_pending(_):
        rows = {r["id"]: r for r in load_reviews() if r["config"] == config}
        for k in range(1, len(ids) + 1):
            j = (state["i"] + k) % len(ids)
            if not is_reviewed(rows[ids[j]]):
                state["i"] = j; show(); return
        status.value += " · все строки оценены"

    b_prev.on_click(lambda _: move(-1)); b_next.on_click(lambda _: move(1))
    b_save.on_click(on_save); b_pending.on_click(to_pending)
    show()
    return W.VBox([
        W.HBox([img, info]),
        correctness,
        W.HBox(list(err_boxes.values())),
        W.HBox([n_claims, n_unsup, reviewer]),
        comment,
        W.HBox([b_prev, b_next, b_pending, b_save]),
        status,
    ])

display(review_form())

# 2. Сколько заполнено и сколько ожидает; отчёт без заполненных строк не выдаёт частот как итоговых
report = build_error_report()
print(f"Просмотрено {report['n_reviewed']} из {report['n_total']}, ожидают оценки {report['n_pending']}")
print(f"Статус отчёта: {report['status']} — {report['note']}")
if report["n_reviewed"]:
    print("Ошибки (по просмотренным):", report["overall"]["error_counts"])
    print("Correctness:", report["overall"]["correctness"])
    print("Hallucination rate H:", report["hallucination_rate_claims"]["H"])

check(ERROR_REPORT_PATH.exists(), f"{ERROR_REPORT_PATH.name} сохранён")
check(report["n_reviewed"] + report["n_pending"] == report["n_total"], "число заполненных и ожидающих строк сходится")
check(report["status"] == "final" or "ПРЕДВАРИТЕЛЬНЫЙ" in report["note"],
      "незавершённая разметка помечена как предварительная, а не итоговая частота ошибок")
check(report["n_pending"] == 0, f"разметка завершена ({report['n_reviewed']}/{report['n_total']})")


Просмотрено 20 из 20, ожидают оценки 0
Статус отчёта: final — все строки просмотрены — частоты итоговые
Ошибки (по просмотренным): {'hallucination': 4, 'ocr': 2, 'spatial': 1, 'counting': 0, 'wrong_object': 4, 'format': 0}
Correctness: {'correct': 11, 'partially_correct': 6, 'incorrect': 3}
Hallucination rate H: 0.1846
✔ error_report.json сохранён
✔ число заполненных и ожидающих строк сходится
✔ незавершённая разметка помечена как предварительная, а не итоговая частота ошибок
✔ разметка завершена (20/20)


### Задание 8. Парная абляция prompt и разрешения

*Вес: 1 балл.*

**Постановка.** Варианты модели сравниваются только на одном наборе изображений.

Выполните следующие действия:
1. Переберите шаблоны, разрешения и режимы декодирования.
2. Инкрементально сохраните реальные ответы для каждой конфигурации.

**Результат.** ablation.jsonl с предсказаниями VLM для каждого варианта.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


**Задание 8. Парная абляция prompt, разрешения и декодирования**

- **Что делали**: на тех же 20 картинках сравнили 4 варианта, каждый отличается от базового ровно одним фактором: шаблон `short_answer`, разрешение 448², sampling. Ответы базового варианта взяли из задания 4, остальные досчитывали с записью после каждого ответа.
- **Зачем**: понять, какая настройка и как влияет на ответы. Если менять всё сразу, причину не определить.
- **Результат**: ablation.jsonl, 80 ответов (20 базовых + 60 новых, ≈ 9,6 мин генерации), без повторов, на одном наборе изображений.


In [11]:
# Задание 8. Парная абляция prompt, разрешения и декодирования
ABLATION_PATH = ARTIFACTS / ("ablation_fast.jsonl" if FAST_DEV_RUN else "ablation.jsonl")

# 1. Контролируемая абляция: от базовой конфигурации меняется ровно один фактор за раз,
#    поэтому разницу в качестве и задержке можно приписать именно ему. Набор изображений у всех вариантов один и тот же
ABLATION_VARIANTS = {
    "baseline":       dict(factor="—",          template="plain",        resolution="default", decoding="greedy"),
    "prompt_short":   dict(factor="prompt",     template="short_answer", resolution="default", decoding="greedy"),
    "resolution_low": dict(factor="resolution", template="plain",        resolution="low",     decoding="greedy"),
    "decoding_sample":dict(factor="decoding",   template="plain",        resolution="default", decoding="sampling"),
}

def variant_config_name(v):
    return "{template}|{resolution}|{decoding}".format(**v)

def is_same_ablation_run(r):
    v = ABLATION_VARIANTS.get(r.get("variant"))
    return (v is not None and r.get("config") == variant_config_name(v)
            and r.get("model_id") == model_id and r.get("revision") == model_revision)

existing = read_jsonl_safe(ABLATION_PATH)
done = {(r["variant"], r["id"]) for r in existing if is_same_ablation_run(r)}

# Базовая конфигурация уже посчитана в задании 4 — переносим эти же реальные ответы, а не генерируем заново
baseline_preds = {p["id"]: p for p in json.loads(PRED_JSON.read_text(encoding="utf-8")) if p["config"] == CONFIG_NAME}
for ex in eval_rows:
    if ("baseline", ex["id"]) not in done and ex["id"] in baseline_preds:
        append_jsonl(ABLATION_PATH, {**baseline_preds[ex["id"]], "variant": "baseline", "factor": "—",
                                     "source": PRED_JSONL.name})
        done.add(("baseline", ex["id"]))

todo = [(name, v, ex) for name, v in ABLATION_VARIANTS.items() for ex in eval_rows if (name, ex["id"]) not in done]
print(f"{ABLATION_PATH.name}: готово {len(done)}/{len(ABLATION_VARIANTS) * len(eval_rows)}, осталось {len(todo)}")

# 2. Инкрементальная запись: каждый ответ сразу на диск, прерванный запуск продолжается без повторов
try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **k: x
failed = []
for name, v, ex in tqdm(todo, desc="абляция"):
    cfg = {k: v[k] for k in ("template", "resolution", "decoding")}
    try:
        res = run_vlm(ex["image"], ex["question"], task=ex["task"], **cfg)
    except torch.cuda.OutOfMemoryError as e:
        torch.cuda.empty_cache()
        failed.append((name, ex["id"])); print(f"  ✘ {name}/{ex['id']}: OOM ({e})"); continue
    append_jsonl(ABLATION_PATH, {
        "id": ex["id"], "task": ex["task"], "category": ex["category"],
        "question": ex["question"], "reference": ex["reference"], "image": ex["image"],
        "variant": name, "factor": v["factor"], "config": variant_config_name(v), **res,
        "source": "ablation_run", "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    })

ablation_rows = [r for r in read_jsonl_safe(ABLATION_PATH) if is_same_ablation_run(r)]
by_variant = {name: {r["id"]: r for r in ablation_rows if r["variant"] == name} for name in ABLATION_VARIANTS}
eval_ids = {ex["id"] for ex in eval_rows}

print(f"\n{'вариант':<17}{'фактор':<12}{'конфигурация':<28}{'n':>3}{'медиана, с':>12}{'ток. (ср.)':>11}{'виз. ток.':>10}")
for name, rows in by_variant.items():
    rs = list(rows.values())
    if rs:
        print(f"{name:<17}{ABLATION_VARIANTS[name]['factor']:<12}{variant_config_name(ABLATION_VARIANTS[name]):<28}{len(rs):>3}"
              f"{np.median([r['latency_s'] for r in rs]):>12.1f}{np.mean([r['n_output_tokens'] for r in rs]):>11.0f}"
              f"{np.mean([r['n_image_tokens'] for r in rs]):>10.0f}")

# Один и тот же пример во всех вариантах — для глаза
demo = next(ex for ex in eval_rows if ex["task"] == "ocr")
print(f"\nПример {demo['id']}: {demo['question']}  (эталон: {demo['reference']})")
for name, rows in by_variant.items():
    if demo["id"] in rows:
        print(f"  {name:<17} → {rows[demo['id']]['raw_output'][:90]!r}")

check(all(set(rows) == eval_ids for rows in by_variant.values()),
      f"все {len(ABLATION_VARIANTS)} варианта посчитаны на одном и том же наборе из {len(eval_ids)} изображений"
      + (f"; не удались: {failed}" if failed else ""))
check(len(ablation_rows) == len({(r["variant"], r["id"]) for r in ablation_rows}), "в ablation.jsonl нет повторов")
check(all(sum(ABLATION_VARIANTS[n][k] != ABLATION_VARIANTS["baseline"][k] for k in ("template", "resolution", "decoding")) == 1
          for n in ABLATION_VARIANTS if n != "baseline"), "каждый вариант отличается от базового ровно одним фактором")
check(all(r["raw_output"].strip() for r in ablation_rows), "все ответы — непустые реальные выходы VLM")


ablation.jsonl: готово 80/80, осталось 0


абляция: 0it [00:00, ?it/s]


вариант          фактор      конфигурация                  n  медиана, с ток. (ср.) виз. ток.
baseline         —           plain|default|greedy         20        11.8        129       810
prompt_short     prompt      short_answer|default|greedy  20         1.5          8       810
resolution_low   resolution  plain|low|greedy             20        11.6        120       189
decoding_sample  decoding    plain|default|sampling       20        12.5        125       810

Пример ocr-0: What is the total estimated cost?  (эталон: $2,500)
  baseline          → 'The total estimated cost is $2,500.'
  prompt_short      → '$2,500'
  resolution_low    → 'The total estimated cost is **$2,500**.'
  decoding_sample   → 'The total estimated cost is $2,500.'
✔ все 4 варианта посчитаны на одном и том же наборе из 20 изображений
✔ в ablation.jsonl нет повторов
✔ каждый вариант отличается от базового ровно одним фактором
✔ все ответы — непустые реальные выходы VLM


**Варианты абляции.** Те же 20 картинок и вопросов, в каждом варианте относительно базового меняется ровно одна настройка.

| Вариант | Что поменяли | Шаблон | Разрешение | Декодирование |
|---|---|---|---|---|
| `baseline` | ничего | plain | default | greedy |
| `prompt_short` | шаблон | **short_answer** | default | greedy |
| `resolution_low` | разрешение | plain | **low** | greedy |
| `decoding_sample` | декодирование | plain | default | **sampling** |

**Шаблон** — это формулировка запроса:
- `plain` — вопрос как есть: «What is the total estimated cost?»;
- `short_answer` — тот же вопрос плюс инструкция «Answer with a single word, number or short phrase. Do not explain.»

**Разрешение** — наибольший размер картинки, который получает модель:
- `default` — площадь не больше 1024×1024 пикселей;
- `low` — не больше 448×448.

Меньше картинка — меньше визуальных токенов (в среднем 810 → 189), но мелкие детали могут потеряться. Картинки меньше лимита не растягиваются.

**Режим декодирования** — правило выбора следующего токена при генерации ответа:
- `greedy` — всегда самый вероятный токен; ответ при повторе не меняется;
- `sampling` — токен выбирается случайно среди вероятных (temperature 0,7, top-p 0,8, top-k 20), поэтому ответы разнообразнее.


### Задание 9. Качество абляционных вариантов

*Вес: 1 балл.*

**Постановка.** Изменение prompt или разрешения следует сопоставлять по измеримому качеству коротких ответов.

Выполните следующие действия:
1. Для VQA/OCR рассчитайте нормализованный exact match по каждому варианту.
2. Сохраните размер и точность каждой группы.

**Результат.** ablation_quality.json с фактической точностью VQA/OCR.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


**Задание 9. Качество абляционных вариантов**

- **Что делали**: для каждого варианта посчитали нормализованный EM (и contains, F1) по OCR, open VQA и обеим вместе, сохранили размер групп и сравнили каждую пару примеров с baseline.
- **Зачем**: сопоставить варианты по измеримому качеству коротких ответов.
- **Результат**: ablation_quality.json, по 5 примеров в группе.
  - `prompt_short`: EM на OCR 0,00 → 1,00, улучшились все 5 пар; на open VQA (открытые вопросы по картинке) F1 упал с 0,31 до 0,04;
  - `resolution_low` и `decoding_sample`: EM без изменений (0,00).


In [12]:
# Задание 9. Качество абляционных вариантов
ABLATION_QUALITY_PATH = ARTIFACTS / ("ablation_quality_fast.json" if FAST_DEV_RUN else "ablation_quality.json")

# Ответы всех вариантов с диска — ячейка работает и после перезапуска ядра (нужны функции из заданий 5 и 8)
ablation_rows = [r for r in read_jsonl_safe(ABLATION_PATH) if is_same_ablation_run(r)]
short_rows = [r for r in ablation_rows if r["task"] in ("vqa", "ocr")]
for r in short_rows:
    r["scores"] = score_prediction(r)   # та же нормализация и тот же EM, что в metrics.json

GROUPS = {"document_qa": lambda r: r["category"] == "document_qa",
          "open_vqa": lambda r: r["category"] == "open_vqa",
          "vqa_ocr": lambda r: True}

def group_stats(rows):
    """Размер группы и точность: EM — основная метрика, contains и token F1 — вспомогательные."""
    n = len(rows)
    return {"n": n,
            "exact_match": round(float(np.mean([r["scores"]["exact_match"] for r in rows])), 4) if n else None,
            "n_exact": int(sum(r["scores"]["exact_match"] for r in rows)),
            "contains_match": round(float(np.mean([r["scores"]["contains_match"] for r in rows])), 4) if n else None,
            "token_f1": round(float(np.mean([r["scores"]["token_f1"] for r in rows])), 4) if n else None,
            "mean_output_tokens": round(float(np.mean([r["n_output_tokens"] for r in rows])), 1) if n else None}

# 1. Нормализованный EM по каждому варианту и группе
per_variant = {}
for name, v in ABLATION_VARIANTS.items():
    rows = [r for r in short_rows if r["variant"] == name]
    per_variant[name] = {"factor": v["factor"], "config": variant_config_name(v),
                         "groups": {g: group_stats([r for r in rows if f(r)]) for g, f in GROUPS.items()}}

# Парное сравнение с baseline на тех же примерах: где EM вырос, где упал
base = {r["id"]: r["scores"]["exact_match"] for r in short_rows if r["variant"] == "baseline"}
for name in ABLATION_VARIANTS:
    if name == "baseline":
        continue
    cur = {r["id"]: r["scores"]["exact_match"] for r in short_rows if r["variant"] == name}
    common = sorted(set(base) & set(cur))
    per_variant[name]["paired_vs_baseline"] = {
        "n_pairs": len(common),
        "delta_exact_match": round(float(np.mean([cur[i] - base[i] for i in common])), 4) if common else None,
        "improved_ids": [i for i in common if cur[i] > base[i]],
        "worsened_ids": [i for i in common if cur[i] < base[i]],
    }

ablation_quality = {
    "model_id": model_id, "revision": model_revision,
    "metric": "нормализованный exact match (та же normalize_answer, что в metrics.json); contains и token F1 — вспомогательные",
    "n_images": len({r["image"] for r in short_rows}),
    "variants": per_variant,
    "per_example": [{"variant": r["variant"], "id": r["id"], "category": r["category"],
                     "pred_normalized": normalize_answer(r["raw_output"]), "ref_normalized": normalize_answer(r["reference"]),
                     **r["scores"]} for r in short_rows],
    "caveat": "по 5 примеров на категорию — различие в 1 ответ даёт 0.2; выводы качественные, не статистически значимые",
    "fast_dev_run": FAST_DEV_RUN,
}
# 2. Размер и точность каждой группы
ABLATION_QUALITY_PATH.write_text(json.dumps(ablation_quality, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"{'вариант':<17}{'фактор':<12}{'группа':<13}{'n':>3}{'EM':>7}{'contains':>10}{'F1':>7}{'ток.':>7}")
for name, info in per_variant.items():
    for g in ("document_qa", "open_vqa", "vqa_ocr"):
        s = info["groups"][g]
        print(f"{name if g == 'document_qa' else '':<17}{info['factor'] if g == 'document_qa' else '':<12}{g:<13}{s['n']:>3}"
              f"{s['exact_match']:>7.2f}{s['contains_match']:>10.2f}{s['token_f1']:>7.2f}{s['mean_output_tokens']:>7.0f}")
print("\nПарно с baseline (VQA+OCR, EM):")
for name, info in per_variant.items():
    if "paired_vs_baseline" in info:
        p = info["paired_vs_baseline"]
        print(f"  {name:<17} Δ EM = {p['delta_exact_match']:+.2f}  лучше: {p['improved_ids']}  хуже: {p['worsened_ids']}")

check(all(info["groups"]["vqa_ocr"]["n"] == len([e for e in eval_rows if e["task"] in ("vqa", "ocr")])
          for info in per_variant.values()), "у всех вариантов одинаковый размер группы VQA/OCR")
check(all(info["groups"][g]["exact_match"] is not None for info in per_variant.values() for g in GROUPS),
      "exact match посчитан для каждой группы каждого варианта")
check(per_variant["baseline"]["groups"]["vqa_ocr"]["exact_match"] == metrics["short_answers_vqa_ocr"]["exact_match"]
      if "metrics" in globals() else True, "baseline совпадает с metrics.json (одна и та же метрика на одних ответах)")
check(ABLATION_QUALITY_PATH.exists(), f"{ABLATION_QUALITY_PATH.name} сохранён")


вариант          фактор      группа         n     EM  contains     F1   ток.
baseline         —           document_qa    5   0.00      0.80   0.31     26
                             open_vqa       5   0.00      0.00   0.31    220
                             vqa_ocr       10   0.00      0.40   0.31    123
prompt_short     prompt      document_qa    5   1.00      1.00   1.00      7
                             open_vqa       5   0.00      0.00   0.04      4
                             vqa_ocr       10   0.50      0.50   0.52      6
resolution_low   resolution  document_qa    5   0.00      0.80   0.31     28
                             open_vqa       5   0.00      0.00   0.32    203
                             vqa_ocr       10   0.00      0.40   0.31    116
decoding_sample  decoding    document_qa    5   0.00      0.80   0.29     30
                             open_vqa       5   0.00      0.00   0.32    198
                             vqa_ocr       10   0.00      0.40   0.30    114

### Задание 10. Задержка и сводная оценка

*Вес: 1 балл.*

**Постановка.** Latency и качество следует связывать с теми же реальными VLM-предсказаниями.

Выполните следующие действия:
1. Вычислите median и p95 задержки каждого варианта.
2. Сохраните сводку качества, ручных ошибок и бюджета.

**Результат.** eval_summary.json с фактическими метриками и числом изображений.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


**Задание 10. Задержка и сводная оценка**

- **Что делали**: по тем же записям `ablation.jsonl` посчитали median и p95 (95-й перцентиль — время, которое не превышают 95% ответов) задержки каждого варианта и по задачам. Собрали сводку качества, ручных ошибок и бюджета.
- **Зачем**: связать скорость и качество на одних и тех же реальных предсказаниях.
- **Результат**: eval_summary.json, 20 изображений.
  - median / p95: baseline 11,8 / 27,4 с, `prompt_short` 1,5 / 2,7 с, `resolution_low` 11,6 / 27,6 с, `decoding_sample` 12,5 / 28,0 с;
  - бюджет: 80 вызовов generate, 14,3 мин, пик памяти 3,92 ГБ, 16 ответов упёрлись в лимит токенов.


In [13]:
# Задание 10. Задержка и сводная оценка
EVAL_SUMMARY_PATH = ARTIFACTS / ("eval_summary_fast.json" if FAST_DEV_RUN else "eval_summary.json")

def load_json(name):
    return json.loads((ARTIFACTS / name).read_text(encoding="utf-8"))

# Всё берём из сохранённых артефактов: задержка и качество считаются по ОДНИМ И ТЕМ ЖЕ записям ablation.jsonl
ablation_rows = [r for r in read_jsonl_safe(ABLATION_PATH) if is_same_ablation_run(r)]
model_card_saved = load_json("model_config.json")
metrics_saved = load_json("metrics.json")
abl_quality_saved = load_json(ABLATION_QUALITY_PATH.name)
error_report_saved = load_json(ERROR_REPORT_PATH.name)

def latency_stats(rows):
    lat = np.array([r["latency_s"] for r in rows], dtype=float)
    out_tok = np.array([r["n_output_tokens"] for r in rows], dtype=float)
    return {"n": int(lat.size),
            "median_s": round(float(np.median(lat)), 3),
            "p95_s": round(float(np.percentile(lat, 95)), 3),  # при n=20 p95 ≈ между 19-м и 20-м значением
            "mean_s": round(float(lat.mean()), 3),
            "max_s": round(float(lat.max()), 3),
            "total_generate_s": round(float(lat.sum()), 1),
            "median_preprocess_s": round(float(np.median([r["preprocess_s"] for r in rows])), 4),
            "tokens_per_s": round(float(out_tok.sum() / lat.sum()), 2),
            "mean_output_tokens": round(float(out_tok.mean()), 1),
            "mean_image_tokens": round(float(np.mean([r["n_image_tokens"] for r in rows])), 1),
            "peak_gpu_mem_gb": max((r["peak_gpu_mem_gb"] or 0) for r in rows)}

# 1. Median и p95 задержки каждого варианта (+ по задачам: описания и подсчёт различаются на порядок)
variants_summary = {}
for name, v in ABLATION_VARIANTS.items():
    rows = [r for r in ablation_rows if r["variant"] == name]
    q = abl_quality_saved["variants"][name]["groups"]
    variants_summary[name] = {
        "factor": v["factor"], "config": variant_config_name(v),
        "latency": latency_stats(rows),
        "latency_by_task": {t: {k: s[k] for k in ("n", "median_s", "p95_s")}
                            for t in sorted({r["task"] for r in rows})
                            for s in [latency_stats([r for r in rows if r["task"] == t])]},
        "quality_vqa_ocr": {g: {k: q[g][k] for k in ("n", "exact_match", "contains_match", "token_f1")} for g in q},
        "ids": sorted(r["id"] for r in rows),
    }

# 2. Сводка: качество, ручные ошибки, бюджет
all_ids = sorted({r["id"] for r in ablation_rows})
eval_summary = {
    "model": {k: model_card_saved[k] for k in ("model_id", "revision", "is_fallback", "compute_dtype",
                                              "memory_footprint_gb", "linear4bit_layers", "gpu", "versions")},
    "quantization": model_card_saved["quantization"]["bnb_4bit_quant_type"] if isinstance(model_card_saved["quantization"], dict) else "none",
    "data": {"n_images": len({r["image"] for r in ablation_rows}), "n_examples": len(all_ids),
             "per_task": dict(Counter(e["task"] for e in eval_rows)),
             "sources": sorted({e["source"] for e in eval_rows})},
    "baseline_quality": {
        "config": metrics_saved["config"],
        "short_answers_vqa_ocr": metrics_saved["short_answers_vqa_ocr"],
        "per_category": metrics_saved["per_category"],
        "caption": metrics_saved["caption"],
    },
    "manual_errors": {k: error_report_saved[k] for k in ("status", "note", "n_total", "n_reviewed", "n_pending")}
                     | {"correctness": error_report_saved["overall"]["correctness"],
                        "error_counts": error_report_saved["overall"]["error_counts"],
                        "error_rates": error_report_saved["overall"]["error_rates"],
                        "hallucination_rate_H": error_report_saved["hallucination_rate_claims"]["H"],
                        "per_task_correctness": {t: v["correctness"] for t, v in error_report_saved["per_task"].items()}},
    "variants": variants_summary,
    "budget": {
        "n_generate_calls": len(ablation_rows),
        "n_new_calls_ablation": sum(r.get("source") == "ablation_run" for r in ablation_rows),
        "total_generate_gpu_min": round(sum(r["latency_s"] for r in ablation_rows) / 60, 1),
        "peak_gpu_mem_gb": max((r["peak_gpu_mem_gb"] or 0) for r in ablation_rows),
        "model_load_s": model_card_saved["load_seconds"],
        "max_new_tokens": {**MAX_NEW_TOKENS, "default": DEFAULT_MAX_NEW_TOKENS},
        "n_hit_max_new_tokens": sum(r["hit_max_new_tokens"] for r in ablation_rows),
        "profile": "VLM 4B NF4, одно изображение на запрос, 20 примеров (5 × 4 задачи)",
    },
    "fast_dev_run": FAST_DEV_RUN,
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
EVAL_SUMMARY_PATH.write_text(json.dumps(eval_summary, ensure_ascii=False, indent=2), encoding="utf-8")

# Задержка и качество — в одной таблице, по тем же предсказаниям
print(f"{'вариант':<17}{'фактор':<12}{'n':>3}{'median, с':>11}{'p95, с':>9}{'ток/с':>7}{'ток.':>6}{'виз.ток.':>9}{'EM OCR':>8}{'EM VQA+OCR':>12}")
for name, s in variants_summary.items():
    L, Q = s["latency"], s["quality_vqa_ocr"]
    print(f"{name:<17}{s['factor']:<12}{L['n']:>3}{L['median_s']:>11.2f}{L['p95_s']:>9.2f}{L['tokens_per_s']:>7.1f}"
          f"{L['mean_output_tokens']:>6.0f}{L['mean_image_tokens']:>9.0f}{Q['document_qa']['exact_match']:>8.2f}{Q['vqa_ocr']['exact_match']:>12.2f}")

print("\nMedian задержки по задачам, с:")
tasks = sorted({r["task"] for r in ablation_rows})
print(f"{'вариант':<17}" + "".join(f"{t:>12}" for t in tasks))
for name, s in variants_summary.items():
    print(f"{name:<17}" + "".join(f"{s['latency_by_task'][t]['median_s']:>12.2f}" for t in tasks))

me, b = eval_summary["manual_errors"], eval_summary["budget"]
print(f"\nРучная разметка ({me['status']}, {me['n_reviewed']}/{me['n_total']}): {me['correctness']}; ошибки {me['error_counts']}; H = {me['hallucination_rate_H']}")
print(f"Бюджет: {b['n_generate_calls']} вызовов generate, {b['total_generate_gpu_min']} мин GPU, пик памяти {b['peak_gpu_mem_gb']} ГБ, "
      f"упёрлись в max_new_tokens: {b['n_hit_max_new_tokens']}")
print(f"Изображений: {eval_summary['data']['n_images']}, примеров: {eval_summary['data']['n_examples']}")

check(all(s["ids"] == all_ids for s in variants_summary.values()), "задержка всех вариантов измерена на одном наборе изображений")
quality_ids = {(e["variant"], e["id"]) for e in abl_quality_saved["per_example"]}
latency_ids = {(r["variant"], r["id"]) for r in ablation_rows if r["task"] in ("vqa", "ocr")}
check(quality_ids == latency_ids, "latency и качество VQA/OCR посчитаны по одним и тем же предсказаниям ablation.jsonl")
check(all(s["latency"]["p95_s"] >= s["latency"]["median_s"] > 0 for s in variants_summary.values()), "median и p95 посчитаны для каждого варианта")
check(me["status"] == "final", f"в сводку вошла завершённая ручная разметка ({me['n_reviewed']}/{me['n_total']})")
check(EVAL_SUMMARY_PATH.exists(), f"{EVAL_SUMMARY_PATH.name} сохранён")


вариант          фактор        n  median, с   p95, с  ток/с  ток. виз.ток.  EM OCR  EM VQA+OCR
baseline         —            20      11.83    27.37    9.2   129      810    0.00        0.00
prompt_short     prompt       20       1.52     2.73    5.1     8      810    1.00        0.50
resolution_low   resolution   20      11.59    27.57    9.2   120      189    0.00        0.00
decoding_sample  decoding     20      12.46    28.05    8.9   125      810    0.00        0.00

Median задержки по задачам, с:
вариант            captioning         ocr  structured         vqa
baseline                27.20        2.68        2.53       26.42
prompt_short             1.27        1.66        2.46        0.70
resolution_low          26.61        2.15        1.99       19.20
decoding_sample         27.70        3.07        2.45       22.70

Ручная разметка (final, 20/20): {'correct': 11, 'partially_correct': 6, 'incorrect': 3}; ошибки {'hallucination': 4, 'ocr': 2, 'spatial': 1, 'counting': 0, 'wrong

### Задание 11 (бонус). Проверка формулировки вопроса

*Вес: 1 дополнительный балл.*

**Постановка.** Уточнение вопроса может изменить ответ модели без смены изображения.

Выполните следующие действия:
1. Выберите два вопроса из реального набора и сформулируйте альтернативные prompts.
2. Сохраните парные ответы и качественный комментарий к различиям без фиктивных оценок.

**Результат.** bonus_prompts.json с исходными и новыми ответами.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


**Задание 11 (бонус). Проверка формулировки вопроса**

- **Что делали**: для двух ошибочных примеров (`vqa-5`, `structured-12`) задали по две альтернативные формулировки, не раскрывающие ответ. Сохранили исходные и новые ответы, проверяемые факты и комментарий без оценок.
- **Зачем**: проверить, исправляет ли уточнение вопроса ошибку при той же картинке.
- **Результат**: bonus_prompts.json, 4 новых ответа. Ни одна формулировка не помогла: `vqa-5` снова Hummer вместо G6, `structured-12` снова 8 или сломанный JSON с 10 вместо 9.


In [16]:
# Задание 11 (бонус). Проверка формулировки вопроса
BONUS_PATH = ARTIFACTS / ("bonus_prompts_fast.json" if FAST_DEV_RUN else "bonus_prompts.json")

# 1. Два реальных вопроса, на которых базовая модель ошиблась (по ручной разметке), и альтернативные формулировки.
#    Картинка, модель, разрешение и декодирование те же, что в базовом прогоне — меняется только текст вопроса.
#    Подсказки не раскрывают ответ: они уточняют, ЧТО спрашивается и КАК смотреть на изображение.
BONUS_ITEMS = {
    "vqa-5": {
        "why": "модель приняла водяной знак студии Humster3D за марку Hummer",
        "key_terms": ["g6", "howitzer", "artillery"],   # термины из эталона — только для фактической пометки, не оценка
        "alternatives": {
            "ignore_watermark": "The text in the corner is a watermark of a 3D-model studio, not the vehicle's brand. "
                                "Ignoring the watermark, what specific military vehicle is this a model of?",
            "describe_then_name": "Look at the vehicle itself: count its wheels and describe its armament and hull. "
                                  "Then name the specific military vehicle this is a model of.",
        },
    },
    "structured-12": {
        "why": "модель насчитала 8 человек вместо 9 в плотной группе",
        "key_terms": [],
        "alternatives": {
            "row_by_row": "Count the people row by row: first the standing back row, then the seated front row. "
                          "Then return only JSON with keys label and count, using label 'people'.",
            "include_partial": "How many people are visible, including anyone partially hidden behind others? "
                               "Return only JSON with keys label and count, using label 'people'.",
        },
    },
}

baseline_by_id = {p["id"]: p for p in json.loads(PRED_JSON.read_text(encoding="utf-8"))}
eval_by_id = {e["id"]: e for e in eval_rows}

def facts(item_id, text):
    """Проверяемые факты об ответе — не оценки качества: число из JSON, упомянутые термины эталона, длина."""
    ex = eval_by_id[item_id]
    out = {"n_chars": len(text)}
    if ex["task"] == "structured":
        got = parse_count_json(text)
        out["parsed_count"] = got.get("count") if got else None
        out["reference_count"] = json.loads(ex["reference"])["count"]
    else:
        low = text.lower()
        out["mentions_reference_terms"] = [t for t in BONUS_ITEMS[item_id]["key_terms"] if t in low]
    return out

# Возобновление: уже полученные новые ответы не пересчитываем
saved = json.loads(BONUS_PATH.read_text(encoding="utf-8")) if BONUS_PATH.exists() else {"items": []}
saved_alt = {(it["id"], a["prompt_name"]): a for it in saved["items"] for a in it["alternatives"]
             if it.get("model_id") == model_id and it.get("revision") == model_revision}
saved_comments = {it["id"]: it.get("comment") for it in saved["items"]}

# Качественный комментарий пишет человек ПОСЛЕ просмотра пар ответов; пустой — значит ещё не написан
BONUS_COMMENTS = {
    "vqa-5": ("Уточнение не исправило ошибку. С подсказкой про водяной знак модель всё равно назвала Hummer "
              "(«M1127 HUMVEE», смешав названия разных машин) — привязка к слову Humster сильнее инструкции. "
              "Просьба сначала описать машину ухудшила и то, что было верно: вместо шести колёс модель "
              "насчитала «восемь» и противоречит сама себе. G6/гаубица не названы ни в одном варианте."),
    "structured-12": ("Переформулировка не помогла. Подсказка про частично заслонённых людей ответ не изменила (снова 8). "
                      "Подсчёт по рядам сломал требуемый формат: вместо JSON с label/count — вложенный список рядов, "
                      "и счёт стал хуже: 5 сзади + 5 спереди = 10 при реальных 4 + 5 = 9. Пошаговая инструкция "
                      "добавила ошибку формата и не исправила подсчёт."),
}

items = []
for item_id, spec in BONUS_ITEMS.items():
    ex, base = eval_by_id[item_id], baseline_by_id[item_id]
    alts = []
    for prompt_name, alt_question in spec["alternatives"].items():
        if (item_id, prompt_name) in saved_alt:
            alts.append(saved_alt[(item_id, prompt_name)]); continue
        res = run_vlm(ex["image"], alt_question, task=ex["task"], **MAIN_CONFIG)
        alts.append({"prompt_name": prompt_name, "question": alt_question, "raw_output": res["raw_output"],
                     "latency_s": res["latency_s"], "n_output_tokens": res["n_output_tokens"],
                     "hit_max_new_tokens": res["hit_max_new_tokens"], "facts": facts(item_id, res["raw_output"])})
    items.append({
        "id": item_id, "task": ex["task"], "image": ex["image"], "reference": ex["reference"],
        "why_selected": spec["why"], "config": CONFIG_NAME, "model_id": model_id, "revision": model_revision,
        "original": {"question": ex["question"], "raw_output": base["raw_output"], "latency_s": base["latency_s"],
                     "facts": facts(item_id, base["raw_output"])},
        "alternatives": alts,
        "comment": BONUS_COMMENTS.get(item_id) or saved_comments.get(item_id) or None,
    })
    # Сохраняем после каждого вопроса — прерывание не теряет полученные ответы
    BONUS_PATH.write_text(json.dumps({"items": items + [it for it in saved["items"] if it["id"] not in {i["id"] for i in items}],
                                      "note": "парные ответы одной модели на одно изображение; оценок нет, только факты и комментарий"},
                                     ensure_ascii=False, indent=2), encoding="utf-8")

for it in items:
    print(f"\n=== {it['id']} ({it['task']}) — {it['why_selected']}\nЭталон: {it['reference'][:150]}")
    print(f"[исходный] {it['original']['question']}\n   → {it['original']['raw_output'][:300]!r}\n   факты: {it['original']['facts']}")
    for a in it["alternatives"]:
        print(f"[{a['prompt_name']}] {a['question']}\n   → {a['raw_output'][:300]!r}\n   факты: {a['facts']}")
    print("Комментарий:", it["comment"] or "— ещё не написан (заполните BONUS_COMMENTS после просмотра)")

check(len(items) == 2 and all(len(it["alternatives"]) >= 1 for it in items), "два реальных вопроса, у каждого есть альтернативные prompts")
check(all(it["original"]["raw_output"] and all(a["raw_output"] for a in it["alternatives"]) for it in items),
      "сохранены исходные и новые реальные ответы VLM")
check(all(it["comment"] for it in items), "качественный комментарий к различиям написан для каждого вопроса")



=== vqa-5 (vqa) — модель приняла водяной знак студии Humster3D за марку Hummer
Эталон: This image shows a model of a G6 howitzer, a self-propelled artillery vehicle. It's a large, eight-wheeled armored fighting vehicle designed for deser
[исходный] What is this a model of in this image?
   → 'This is a 3D model of an **AFV (Armored Fighting Vehicle)**, specifically a **Hummer** variant, as indicated by the “HUMSTER3D #050” watermark in the top-left corner.\n\nThe vehicle shown is a **six-wheeled, armored, military-style vehicle** with a rugged, boxy design, likely representing a **Hummer H'
   факты: {'n_chars': 1000, 'mentions_reference_terms': []}
[ignore_watermark] The text in the corner is a watermark of a 3D-model studio, not the vehicle's brand. Ignoring the watermark, what specific military vehicle is this a model of?
   → 'This is a model of the **M1127 HUMVEE** (also known as the **HUMVEE** or **“Hummer”**).\n\nThe vehicle shown is a **six-wheel-drive, armored tactical vehicl

**Что мы сделали.** Проверили, можно ли исправить ошибку модели, просто переформулировав вопрос. Картинка, модель и все настройки остались прежними, менялся только текст запроса.

1. Выбрали два вопроса, на которых модель ошиблась: `vqa-5` и `structured-12`.
2. Для каждого придумали по две альтернативные формулировки. Подсказки не раскрывают ответ, а только уточняют, как смотреть на картинку.
3. Сохранили в `artifacts/lab6/bonus_prompts.json` исходные и новые ответы.
4. Добавили к ним проверяемые факты и качественный комментарий, без баллов и фиктивных оценок.

**Результат:**

| Пример | Формулировка | Что ответила модель |
|---|---|---|
| `vqa-5`: модель гаубицы G6 | исходная | Hummer |
| | `ignore_watermark`: не учитывай водяной знак | снова Hummer («M1127 HUMVEE») |
| | `describe_then_name`: сначала опиши, потом назови | вместо верных 6 колёс стало «8», G6 не названа |
| `structured-12`: на фото 9 человек | исходная | 8 |
| | `row_by_row`: считай по рядам | сломан формат JSON, 5 + 5 = 10 |
| | `include_partial`: включая заслонённых | снова 8 |

**Вывод.** Ни одна переформулировка не исправила ошибку, а две сделали ответ хуже. Если модель не знает объект или неточно считает, формулировка этого не лечит. В задании 9 инструкция «отвечай коротко» подняла EM потому, что меняла только форму ответа, а не знания модели.


## Анализ результатов

Сопоставьте измеренные показатели на отложенных реальных данных. Укажите бюджет, ошибки модели и ограничения сокращённого запуска.


**Данные и модель.** 20 реальных примеров PixMo (по 5 на captioning, open VQA, document QA и подсчёт в JSON) с опубликованными эталонами и происхождением (`eval.jsonl`). Модель — Qwen3-VL-4B-Instruct (ревизия `ebb281ec`), языковая часть в NF4 (252 слоя `Linear4bit`), visual encoder в bf16; веса занимают 3,19 ГБ на RTX 3060 12 ГБ. Основная конфигурация — `plain | ≤1024² | greedy`.

**Качество базовой конфигурации** (`metrics.json`, `error_report.json` — ручная разметка 20/20, статус `final`):

| Задача | Автометрики | Ручная оценка (correct / partial / incorrect) |
|---|---|---|
| Document QA (OCR) | EM 0,00 · contains 0,80 · F1 0,31 | **5 / 0 / 0** |
| Подсчёт (JSON) | JSON валиден 1,00 · метка 1,00 · count точно 0,80 · ср. ошибка 0,2 | 4 / 0 / 1 |
| Open VQA | EM 0,00 · contains 0,00 · F1 0,31 | 0 / 3 / 2 |
| Captioning | не оценивается автоматически | 2 / 3 / 0 |
| **Всего** | | **11 / 6 / 3** |

Exact match с шаблоном `plain` равен 0 даже там, где ручная проверка даёт 5/5 верных ответов: модель отвечает полным предложением в Markdown («The total estimated cost is **$2,500**»), а эталон — `$2,500`. Метрика измеряет формат, а не знание.

**Типология ошибок** (ручная, 20 примеров): hallucination 4, wrong_object 4, ocr 2, spatial 1, format 0. Доля неподтверждённых утверждений **H = 24 / 130 ≈ 0,18**. Ошибки сосредоточены в открытых VQA: модель не узнала NASA Super Guppy («NASA DC-9 / Boeing 707» с выдуманным бортовым номером N707NA) и гаубицу G6 (приняла водяной знак студии Humster3D за марку Hummer). В VQA также две OCR-неточности: «25 mm» вместо «125 mm» на объективе, «FRESH POPCORN» вместо «FRESH BUTTERED POP CORN». Единственный промах подсчёта — 8 человек вместо 9 в плотной группе с перекрытиями (`structured-12`). Документы (5/5) и расшифровка таблички на наклонном фото (`captioning-5`) прочитаны без ошибок.

**Абляция** (`ablation.jsonl`, `ablation_quality.json`, `eval_summary.json`; один набор из 20 изображений, меняется ровно один фактор):

| Вариант | Фактор | EM OCR | EM VQA+OCR | F1 open VQA | Median, с | p95, с | Токенов ответа | Визуальных токенов |
|---|---|---|---|---|---|---|---|---|
| baseline | — | 0,00 | 0,00 | 0,31 | 11,8 | 27,4 | 129 | 810 |
| prompt_short | шаблон | **1,00** | **0,50** | 0,04 | **1,5** | **2,7** | 8 | 810 |
| resolution_low | ≤448² | 0,00 | 0,00 | 0,32 | 11,6 | 27,6 | 120 | **189** |
| decoding_sample | sampling | 0,00 | 0,00 | 0,32 | 12,5 | 28,0 | 125 | 810 |

- **Шаблон** «ответь одним словом/числом» поднял EM на OCR с 0 до 1,00 (все 5 пар улучшились) и сократил задержку в ~8 раз. На open VQA короткий ответ теряет содержание («aircraft», «armored vehicle»), поэтому F1 падает до 0,04.
- **Разрешение 448²** сократило визуальные токены в 4,3 раза и пик памяти с 3,91 до 3,39 ГБ, но median почти не изменился: время определяется генерацией текста (~9 токенов/с), а не обработкой изображения. На документах EM и contains не изменились — текст в этих документах крупный.
- **Sampling** (T = 0,7, top-p 0,8) меняет формулировки, но не метрики; на `vqa-2` модель сменила один неверный ответ на другой.

**Бонус** (`bonus_prompts.json`): уточнение формулировки не исправило ни одну ошибку. Подсказка «водяной знак — не марка» всё равно дала Hummer; подсчёт «по рядам» сломал формат JSON и дал 10 вместо 9.

**Бюджет.** 80 вызовов `generate` (20 базовых + 60 новых в абляции, ещё 4 в бонусе), 14,3 мин чистого времени генерации, пик памяти 3,92 ГБ, загрузка модели ≈ 17 с. Описания и open VQA занимают ~27 с на пример, OCR и подсчёт — 2–3 с.

**Ограничения.**
- По 5 примеров на категорию: один ответ меняет точность на 0,2, а p95 при n = 20 почти совпадает с максимумом. Различия качественные, статистически не значимые.
- 16 из 80 ответов упёрлись в `max_new_tokens` (256 для описаний и VQA), поэтому концы описаний не оценены.
- Эталоны сами содержат ошибки, найденные при ручной проверке: «paratrike с парашютом» вместо мотодельтаплана (`vqa-2`), «восьмиколёсная» вместо 6×6 (`vqa-5`), белая стена вместо голубой (`captioning-0`), «coffee/IOS» вместо Kofi/TOS (`captioning-2`). Автометрики наказывают модель за расхождение с такими эталонами.
- Разметку выполнял один эксперт, без второй независимой оценки; число «утверждений» для H подсчитано приближённо.
- Captioning и VQA взяты из train-сплитов PixMo — нельзя исключить, что модель видела их при обучении; документы и подсчёт — из validation.
- Sampling запущен с одним seed, разброс между запусками не измерен.


## Выводы

Сформулируйте предметный вывод по сохранённым артефактам.


1. **Пайплайн собран и воспроизводим:** Qwen3-VL-4B в NF4 укладывается в 3,2 ГБ и работает на RTX 3060; набор с происхождением, конфигурации, сырые ответы, метрики и ручная разметка сохранены в `artifacts/lab6/`, все длительные этапы возобновляются без повторов.
2. **Чтение документов — сильная сторона модели:** 5 из 5 верных ответов по ручной проверке; с коротким шаблоном это подтверждается и строгим EM = 1,00. Подсчёт объектов в JSON — 4 из 5, формат соблюдён всегда.
3. **Главная слабость — распознавание редких объектов:** на open VQA ни одного полностью верного ответа (0 / 3 / 2). Модель уверенно подставляет неверное название по случайной подсказке (Humster → Hummer, NASA → DC-9) и достраивает несуществующие детали; доля неподтверждённых утверждений H ≈ 0,18.
4. **Exact match без контроля формата вводит в заблуждение:** при `plain` он равен 0 при фактически верных ответах. Шаблон ответа — самый сильный из проверенных факторов: он определяет и измеряемую точность коротких ответов, и задержку (median 11,8 → 1,5 с), но для открытых вопросов вредит содержанию.
5. **Снижение разрешения до 448² безопасно для крупного текста** и экономит визуальные токены и память, но почти не ускоряет ответ — задержку VLM определяет длина генерируемого текста. Sampling на этих данных качество не меняет.
6. **Переформулировка вопроса не исправляет ошибок знания:** подсказки не помогли ни в распознавании G6, ни в подсчёте и даже нарушили формат ответа.
7. **Ручная проверка по изображению обязательна:** она нашла ошибки как модели, так и опубликованных эталонов. На 20 примерах выводы качественные; для количественных утверждений нужна выборка больше и независимая вторая разметка.
